# Flipkart Order Issue Resolution Dashboard  
## Python Developer Workplace Handover Capstone

You have joined the simulated Flipkart customer support operations engineering team as a **Python Developer**.

This is not a normal classroom assignment. You are taking ownership of an unfinished internal Colab product after the previous developer resigned. The Product Manager has already written a PRD, the Engineering Manager has handed over the broken codebase, and only one core feature is currently working.

**Your final deliverable:** one completed Google Colab notebook with all outputs visible.

## Project Summary & GitHub Repository

**Project Summary:**

This notebook completes an inherited internal Flipkart operations tool: the *Order Issue Resolution Dashboard*. The previous developer had only finished the basic `orders.csv` loading feature; everything else — multi-file loading, data validation, cleaning, delay/refund/complaint/ticket analysis, priority scoring, recommended actions, AI-ready prompt generation, a Colab search/filter GUI, and final report export — was broken, incomplete, or used the wrong column names.

Working section by section against the PRD and the engineering meeting transcript, I repaired the multi-file loader (wrong filenames, missing Excel/JSON/TXT handling), rebuilt data validation so quality issues are flagged and counted rather than silently dropped, fixed the delay-detection logic to use the correct `actual_delivery_date` column, and — most importantly — restructured the final merge so that refunds, complaints, and support tickets are aggregated to one row per order **before** being merged into the master table, which prevents the accidental row multiplication the PRD specifically warns against. On top of that feature table, I built a transparent, multi-signal priority score (Critical/High/Medium/Low) with documented point values and explicit reasons per order, a recommended-action generator that reacts to the actual combination of signals on each order rather than returning a generic message, and a fact-only AI-ready prompt generator that never calls an external API. The notebook ends with a searchable Colab GUI, a final CSV export matching the required 18-column schema, and 12 automated validation checks that must all pass before the notebook reports success.

All data used is synthetic and created only for this educational exercise; it does not represent real Flipkart operations or customers.

**GitHub Repository:** [PASTE YOUR GITHUB REPO URL HERE BEFORE FINAL SUBMISSION]

*(Replace the line above with your actual repo link, e.g. `https://github.com/<your-username>/flipkart-order-issue-resolution-dashboard`, after you've pushed this notebook. See the accompanying `GitHub_Repo_Setup_Guide.md` for step-by-step instructions.)*


## Handover Context

You are expected to behave like the Python Developer who has inherited this notebook.

The business team needs an internal tool that can:

- Load synthetic order operations datasets.
- Detect delayed and risky orders.
- Identify refund-priority cases.
- Summarize customer complaints and support tickets.
- Produce a final issue-resolution report.
- Provide a simple Colab-based search/filter experience.
- Document implementation decisions, assumptions, PRD mapping, and AI prompt usage.

The previous developer completed only the basic `orders.csv` loading feature. The rest of the notebook contains broken code, incomplete logic, fragile assumptions, and TODO sections.

## Documents You Must Read Before Coding

Before completing the notebook, read the documents provided in the student resource folder:

1. `Meeting_Transcript_Flipkart_Order_Issue_Resolution_Dashboard.docx`
2. `PRD_Flipkart_Order_Issue_Resolution_Dashboard.docx`
3. `prd_completion_checklist.csv`
4. `student_readme.md`

Use the PRD as the product source of truth. Use the meeting transcript as the handover context.

## Dataset Upload Instructions for Google Colab

Upload the dataset ZIP file named:

`flipkart_order_issue_resolution_dataset.zip`

The code below extracts the ZIP and creates the dataset path.

The ZIP contains synthetic training data only. It does not contain real Flipkart data.

In [1]:
from pathlib import Path
import zipfile
import os

DATASET_ZIP = "flipkart_order_issue_resolution_dataset.zip"
EXPECTED_FOLDER = "flipkart_order_issue_resolution_dataset"

# In Google Colab, upload the dataset ZIP if it is not already present.
try:
    from google.colab import files
    if not Path(DATASET_ZIP).exists():
        print("Upload the dataset ZIP file:", DATASET_ZIP)
        uploaded = files.upload()
except Exception:
    # Non-Colab execution fallback.
    pass

if not Path(DATASET_ZIP).exists():
    raise FileNotFoundError(
        f"{DATASET_ZIP} not found. Upload it to the current Colab session before running this notebook."
    )

if not Path(EXPECTED_FOLDER).exists():
    with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
        zip_ref.extractall(".")

DATA_DIR = Path(EXPECTED_FOLDER)
if not (DATA_DIR / "orders.csv").exists():
    # Defensive fallback in case the ZIP structure changes.
    candidates = list(Path(".").rglob("orders.csv"))
    if not candidates:
        raise FileNotFoundError("orders.csv was not found after extracting the dataset ZIP.")
    DATA_DIR = candidates[0].parent

print("Dataset folder ready:", DATA_DIR.resolve())
print("Available files:")
for file in sorted(DATA_DIR.iterdir()):
    print(" -", file.name)

Dataset folder ready: /home/claude/build/flipkart_order_issue_resolution_dataset
Available files:
 - customer_complaints.json
 - customers.csv
 - orders.csv
 - refunds.xlsx
 - sellers.csv
 - status_notes.txt
 - support_tickets.csv


## Section 1: Imports, Configuration, and Developer Environment Check

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`

**How to approach:**
- Review existing imports.
- Add missing imports only when required.
- Confirm the notebook is running in a Colab-compatible Python environment.

**Expected output format:**
- Printed project name.
- Confirmation that Python/Pandas environment started.

**Hint:**
- Keep imports centralized so later sections are easier to debug.

In [2]:
# SECTION 1 — Imports and configuration (repaired)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import json as json_lib
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

PROJECT_NAME = "Flipkart Order Issue Resolution Dashboard"

# Reference "current date" used anywhere a pending record needs its age measured
# against "today". Documented in the Assumption and Limitation Log (Section: Assumption Log).
REFERENCE_DATE = pd.Timestamp("2026-06-30")

# SLA thresholds used consistently across the notebook (documented, not hidden).
REFUND_SLA_DAYS = 7
TICKET_SLA_HOURS = 24

print(PROJECT_NAME)
print("Python/Pandas environment started.")
print("Reference date for pending-case age calculations:", REFERENCE_DATE.date())


Flipkart Order Issue Resolution Dashboard
Python/Pandas environment started.
Reference date for pending-case age calculations: 2026-06-30


## Section 2: Load Main Orders Dataset — Working Core Feature

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`

**How to approach:**
- Use the dataset path created during ZIP extraction.
- Load the main order table with Pandas.
- Display the row count and sample rows.

**Expected output format:**
- Printed row count.
- DataFrame preview with order columns.

**Hint:**
- This is the only feature that the previous developer completed reliably.

In [3]:
# SECTION 2 — Working core feature from the previous developer
# This is the only section that is known to work correctly.

orders_df = pd.read_csv(DATA_DIR / "orders.csv")

print("Orders loaded successfully.")
print("Row count:", len(orders_df))
display(orders_df.head())

Orders loaded successfully.
Row count: 5000


,order_id,customer_id,seller_id,order_date,order_time,promised_delivery_date,actual_delivery_date,order_amount,payment_method,order_status,delay_reason,delivery_city,device_type,seller_category
0,ORD0000001,CUST000771,SELL0475,2026-05-30,10:51:46,2026-06-03,2026-06-03,57056.16,Cash on Delivery,Delivered,NaN,Delhi,iOS,Electronics
1,ORD0000002,CUST000122,SELL0035,2026-05-05,15:59:45,2026-05-08,2026-05-07,25038.34,UPI,Delivered,NaN,Gurugram,iOS,Grocery
2,ORD0000003,CUST000784,SELL0785,2026-05-11,11:48:01,2026-05-15,NaN,29439.12,UPI,processing,Warehouse Delay,Lucknow,Android,Grocery
3,ORD0000004,CUST000288,SELL0211,2026-05-23,03:12:27,2026-05-27,2026-05-25,60697.51,Card,Delivered,NaN,Indore,Android,Beauty
4,ORD0000005,CUST001168,SELL0981,2026-06-14,00:30:13,2026-06-19,NaN,56251.50,UPI,Cancelled,Payment Failed,Nagpur,Mobile Web,Beauty


## Section 3: Basic Order Summary

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`

**How to approach:**
- Summarize rows, columns, unique orders, customers, sellers.
- Find duplicate order IDs and missing values.
- Create a status summary.

**Expected output format:**
- One summary table.
- One order status frequency table.

**Hint:**
- The raw column is `order_status`, not `status`.

In [4]:
# SECTION 3 — Basic order summary (repaired)

order_summary = pd.DataFrame({
    "metric": [
        "row_count", "column_count", "unique_order_count", "duplicate_order_count",
        "unique_customer_count", "unique_seller_count", "total_missing_values",
    ],
    "value": [
        len(orders_df),
        orders_df.shape[1],
        orders_df["order_id"].nunique(),
        int(orders_df["order_id"].duplicated().sum()),
        orders_df["customer_id"].nunique(),
        orders_df["seller_id"].nunique(),
        int(orders_df.isna().sum().sum()),
    ],
})

# The correct raw column is order_status, not status.
order_status_summary = (
    orders_df["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status_raw")
    .reset_index(name="count")
)

print("Order summary:")
display(order_summary)
print("Order status frequency (raw values — note casing/whitespace variants, cleaned later):")
display(order_status_summary)


Order summary:


,metric,value
0,row_count,5000
1,column_count,14
2,unique_order_count,4984
3,duplicate_order_count,16
4,unique_customer_count,1227
5,unique_seller_count,1017
6,total_missing_values,3783


Order status frequency (raw values — note casing/whitespace variants, cleaned later):


,order_status_raw,count
0,Delivered,2684
1,Delayed,577
2,Cancelled,475
3,Processing,423
4,Returned,370
5,Refund Pending,253
6,Lost,83
7,delivered,29
8,DELIVERED,23
9,Delivered,18


## Section 4: Debug Fix Log Setup

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `Notebook codebase`

**How to approach:**
- Create a structured debug log.
- Add a helper function to append fixes.
- Update the log after each major repaired section.

**Expected output format:**
- DataFrame with section, issue_found, fix_applied, validation_output.

**Hint:**
- Think like a developer documenting a handover fix, not like a student writing comments.

In [5]:
# SECTION 4 — Debug fix log setup (repaired)

debug_log = []

def add_debug_fix(section, issue_found, fix_applied, validation_output):
    """Append one structured debug/fix entry documenting an inherited-code repair."""
    debug_log.append({
        "section": section,
        "issue_found": issue_found,
        "fix_applied": fix_applied,
        "validation_output": validation_output,
    })

add_debug_fix(
    section="Section 3 — Basic Order Summary",
    issue_found="Code referenced a non-existent 'status' column instead of 'order_status'; no summary table was built.",
    fix_applied="Rebuilt order_summary and order_status_summary using the real 'order_status' column and added row/column/unique/duplicate/missing counts.",
    validation_output=f"order_summary has {len(order_summary)} metrics; order_status_summary covers {order_status_summary['count'].sum()} orders.",
)

pd.DataFrame(debug_log)


,section,issue_found,fix_applied,validation_output
0,Section 3 — Basic Order Summary,Code referenced a non-existent 'status' column...,Rebuilt order_summary and order_status_summary...,order_summary has 7 metrics; order_status_summ...


## Section 5: Multi-File DataLoader

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`
- `customers.csv`
- `sellers.csv`
- `refunds.xlsx`
- `customer_complaints.json`
- `support_tickets.csv`
- `status_notes.txt`

**How to approach:**
- Create a reusable loader class.
- Use the correct method for each file format.
- Add basic exception handling.

**Expected output format:**
- Summary table with dataset name, rows, and columns/text lines.

**Hint:**
- The refunds file is Excel, complaints are JSON, and notes are TXT.

In [6]:
# SECTION 5 — Multi-file DataLoader (repaired)

class FlipkartIssueDataLoader:
    """Loads every raw dataset file for the issue-resolution dashboard.

    Handles CSV, Excel, JSON, and TXT formats with basic exception handling,
    and returns everything in a single reusable dictionary.
    """

    FILES = {
        "orders": ("orders.csv", "csv"),
        "customers": ("customers.csv", "csv"),
        "sellers": ("sellers.csv", "csv"),
        "refunds": ("refunds.xlsx", "excel"),
        "complaints": ("customer_complaints.json", "json"),
        "tickets": ("support_tickets.csv", "csv"),
        "notes": ("status_notes.txt", "txt"),
    }

    def __init__(self, folder_path):
        self.folder_path = Path(folder_path)
        self.load_status = {}

    def load_dataset_file(self, filename, file_format):
        """Load a single file of a given format, with exception handling."""
        file_path = self.folder_path / filename
        try:
            if file_format == "csv":
                obj = pd.read_csv(file_path)
            elif file_format == "excel":
                obj = pd.read_excel(file_path)
            elif file_format == "json":
                with open(file_path, "r", encoding="utf-8") as f:
                    raw = json_lib.load(f)
                obj = pd.DataFrame(raw)
            elif file_format == "txt":
                with open(file_path, "r", encoding="utf-8") as f:
                    obj = f.readlines()
            else:
                raise ValueError(f"Unsupported format: {file_format}")
            self.load_status[filename] = "Loaded"
            return obj
        except Exception as exc:
            self.load_status[filename] = f"Failed: {exc}"
            return None

    def load_all(self):
        data = {}
        for key, (filename, fmt) in self.FILES.items():
            data[key] = self.load_dataset_file(filename, fmt)
        return data


loader = FlipkartIssueDataLoader(DATA_DIR)
all_data = loader.load_all()

summary_rows = []
for key, (filename, fmt) in FlipkartIssueDataLoader.FILES.items():
    obj = all_data[key]
    if obj is None:
        rows_or_lines, cols, status = 0, 0, loader.load_status[filename]
    elif fmt == "txt":
        rows_or_lines, cols, status = len(obj), 1, loader.load_status[filename]
    else:
        rows_or_lines, cols, status = len(obj), obj.shape[1], loader.load_status[filename]
    summary_rows.append({
        "dataset_name": key,
        "rows_or_lines": rows_or_lines,
        "columns": cols,
        "status": status,
    })

dataset_load_summary = pd.DataFrame(summary_rows)

add_debug_fix(
    section="Section 5 — Multi-File DataLoader",
    issue_found="Loader used wrong filenames ('customer.csv', 'seller.csv'), tried to read refunds.xlsx as CSV, and never loaded complaints JSON or the TXT notes file.",
    fix_applied="Corrected filenames to customers.csv/sellers.csv, added Excel reader for refunds.xlsx, JSON reader for complaints, and a TXT line reader for status_notes.txt, all wrapped in exception handling.",
    validation_output=f"All {len(dataset_load_summary)} files reported status 'Loaded': {(dataset_load_summary['status']=='Loaded').all()}",
)

print("Loaded dataset keys:", list(all_data.keys()))
display(dataset_load_summary)


Loaded dataset keys: ['orders', 'customers', 'sellers', 'refunds', 'complaints', 'tickets', 'notes']


,dataset_name,rows_or_lines,columns,status
0,orders,5000,14,Loaded
1,customers,1200,6,Loaded
2,sellers,1000,6,Loaded
3,refunds,1300,8,Loaded
4,complaints,1500,8,Loaded
5,tickets,1500,8,Loaded
6,notes,100,1,Loaded


## Section 6: Data Validation and Relationship Checks

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`
- `customers.csv`
- `sellers.csv`
- `refunds.xlsx`
- `customer_complaints.json`
- `support_tickets.csv`

**How to approach:**
- Check required columns.
- Check duplicate order IDs.
- Check customer/seller/order relationships.
- Flag invalid dates and suspicious amounts.

**Expected output format:**
- Validation summary table with check_name, status, issue_count, notes.

**Hint:**
- Do not delete invalid records silently; flag them as review items.

In [7]:
# SECTION 6 — Data validation and relationship checks (repaired)

def validate_required_columns(df, required_columns, dataset_name):
    missing = [c for c in required_columns if c not in df.columns]
    return missing

required_columns_map = {
    "orders": (all_data["orders"], ["order_id", "customer_id", "seller_id", "order_amount", "order_status",
                                     "promised_delivery_date", "actual_delivery_date", "delivery_city"]),
    "customers": (all_data["customers"], ["customer_id", "customer_segment"]),
    "sellers": (all_data["sellers"], ["seller_id", "seller_city"]),
    "refunds": (all_data["refunds"], ["refund_id", "order_id", "refund_status", "refund_amount",
                                       "refund_initiated_date", "refund_completed_date"]),
    "complaints": (all_data["complaints"], ["complaint_id", "order_id", "customer_id", "sentiment_tag", "complaint_status"]),
    "tickets": (all_data["tickets"], ["ticket_id", "order_id", "resolution_time_hours", "escalation_flag", "ticket_status"]),
}

validation_rows = []

for name, (df, req_cols) in required_columns_map.items():
    missing = validate_required_columns(df, req_cols, name)
    validation_rows.append({
        "check_name": f"required_columns_{name}",
        "status": "PASS" if not missing else "FAIL",
        "issue_count": len(missing),
        "notes": "All required columns present." if not missing else f"Missing: {missing}",
    })

# Duplicate ID checks
dup_checks = [
    ("duplicate_order_ids", all_data["orders"], "order_id"),
    ("duplicate_customer_ids", all_data["customers"], "customer_id"),
    ("duplicate_seller_ids", all_data["sellers"], "seller_id"),
    ("duplicate_refund_ids", all_data["refunds"], "refund_id"),
    ("duplicate_complaint_ids", all_data["complaints"], "complaint_id"),
    ("duplicate_ticket_ids", all_data["tickets"], "ticket_id"),
]
for check_name, df, col in dup_checks:
    dup_count = int(df[col].duplicated().sum())
    validation_rows.append({
        "check_name": check_name,
        "status": "PASS" if dup_count == 0 else "FLAGGED",
        "issue_count": dup_count,
        "notes": f"{dup_count} duplicate {col} values found (not removed here).",
    })

# Relationship / orphan checks
orphan_checks = [
    ("order_to_customer_orphans", all_data["orders"]["customer_id"], all_data["customers"]["customer_id"]),
    ("order_to_seller_orphans", all_data["orders"]["seller_id"], all_data["sellers"]["seller_id"]),
    ("refund_to_order_orphans", all_data["refunds"]["order_id"], all_data["orders"]["order_id"]),
    ("complaint_to_order_orphans", all_data["complaints"]["order_id"], all_data["orders"]["order_id"]),
    ("ticket_to_order_orphans", all_data["tickets"]["order_id"], all_data["orders"]["order_id"]),
]
for check_name, child_col, parent_col in orphan_checks:
    orphan_count = int((~child_col.isin(parent_col)).sum())
    validation_rows.append({
        "check_name": check_name,
        "status": "PASS" if orphan_count == 0 else "FLAGGED",
        "issue_count": orphan_count,
        "notes": f"{orphan_count} records reference an ID with no matching parent record.",
    })

# Date validity checks
date_checks = [
    ("orders", "order_date"), ("orders", "promised_delivery_date"), ("orders", "actual_delivery_date"),
    ("refunds", "refund_initiated_date"), ("refunds", "refund_completed_date"),
    ("complaints", "complaint_date"), ("tickets", "ticket_created_date"),
]
for dataset_name, col in date_checks:
    df = all_data[dataset_name]
    parsed = pd.to_datetime(df[col], errors="coerce")
    invalid_count = int(parsed.isna().sum() - df[col].isna().sum())  # unparsable but not originally blank
    missing_count = int(df[col].isna().sum())
    validation_rows.append({
        "check_name": f"invalid_or_missing_dates_{dataset_name}_{col}",
        "status": "PASS" if (invalid_count == 0 and missing_count == 0) else "FLAGGED",
        "issue_count": invalid_count + missing_count,
        "notes": f"{missing_count} missing, {invalid_count} unparsable text values.",
    })

# Amount anomaly checks
neg_refund = int((all_data["refunds"]["refund_amount"] < 0).sum())
validation_rows.append({
    "check_name": "negative_refund_amounts",
    "status": "PASS" if neg_refund == 0 else "FLAGGED",
    "issue_count": neg_refund,
    "notes": "Refund records with a negative refund_amount.",
})

refund_order_join = all_data["refunds"].merge(
    all_data["orders"][["order_id", "order_amount"]], on="order_id", how="left"
)
mismatch = int((refund_order_join["refund_amount"] > refund_order_join["order_amount"]).sum())
validation_rows.append({
    "check_name": "refund_amount_exceeds_order_amount",
    "status": "PASS" if mismatch == 0 else "FLAGGED",
    "issue_count": mismatch,
    "notes": "Refund amount greater than the order's order_amount.",
})

validation_summary = pd.DataFrame(validation_rows)

add_debug_fix(
    section="Section 6 — Data Validation",
    issue_found="Validation checked a hard-coded, wrong column list (cust_id, seller, amount, status) and never checked duplicates, relationships, dates, or amount anomalies.",
    fix_applied="Rebuilt validation using real column names and added required-column, duplicate-ID, orphan-relationship, date-validity, and amount-anomaly checks — all flagged, none deleted.",
    validation_output=f"{len(validation_summary)} checks run; {(validation_summary['status']=='FLAGGED').sum()} flagged for review, {(validation_summary['status']=='PASS').sum()} passed.",
)

display(validation_summary)


,check_name,status,issue_count,notes
0,required_columns_orders,PASS,0,All required columns present.
1,required_columns_customers,PASS,0,All required columns present.
2,required_columns_sellers,PASS,0,All required columns present.
3,required_columns_refunds,PASS,0,All required columns present.
4,required_columns_complaints,PASS,0,All required columns present.
5,required_columns_tickets,PASS,0,All required columns present.
6,duplicate_order_ids,FLAGGED,16,16 duplicate order_id values found (not remove...
7,duplicate_customer_ids,PASS,0,0 duplicate customer_id values found (not remo...
8,duplicate_seller_ids,PASS,0,0 duplicate seller_id values found (not remove...
9,duplicate_refund_ids,PASS,0,0 duplicate refund_id values found (not remove...


## Section 7: Data Cleaning and Standardization

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `All loaded datasets`

**How to approach:**
- Standardize text statuses.
- Trim whitespace and normalize casing.
- Parse dates using safe coercion.
- Convert amount and time columns to numeric.

**Expected output format:**
- Cleaning summary table.
- Cleaned DataFrame preview.

**Hint:**
- Use helper functions so cleaning logic is not repeated across every dataset.

In [8]:
# SECTION 7 — Data cleaning and standardization (repaired)

def standardize_text_columns(df, columns):
    """Trim whitespace and title-case the given text columns; returns a copy."""
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()
            df[col] = df[col].replace({"Nan": np.nan, "None": np.nan})
    return df

cleaning_log = []

def log_cleaning(dataset, field, action, affected_records):
    cleaning_log.append({
        "dataset": dataset, "field/type": field, "action": action, "affected_records": affected_records,
    })

# ---- Orders ----
orders_clean = all_data["orders"].copy()
before = orders_clean["order_status"].copy()
orders_clean = standardize_text_columns(orders_clean, ["order_status", "payment_method", "delivery_city",
                                                        "device_type", "seller_category", "delay_reason"])
orders_clean = orders_clean.rename(columns={"order_status": "order_status_clean"})
orders_clean["order_status_clean"] = orders_clean["order_status_clean"].replace({"Refund_Pending": "Refund Pending"})
log_cleaning("orders", "order_status", "trimmed whitespace, standardized casing -> order_status_clean",
             int((before.astype(str).str.strip().str.title() != before).sum()))

for col in ["order_date", "promised_delivery_date", "actual_delivery_date"]:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors="coerce")
log_cleaning("orders", "date columns", "parsed with errors='coerce'", int(orders_clean[["order_date","promised_delivery_date","actual_delivery_date"]].isna().sum().sum()))

orders_clean["order_amount"] = pd.to_numeric(orders_clean["order_amount"], errors="coerce")
log_cleaning("orders", "order_amount", "coerced to numeric", int(orders_clean["order_amount"].isna().sum()))

# ---- Customers ----
customers_clean = all_data["customers"].copy()
customers_clean = standardize_text_columns(customers_clean, ["customer_segment", "preferred_payment_method"])
log_cleaning("customers", "customer_segment/payment", "trimmed + standardized casing", len(customers_clean))

# ---- Sellers ----
sellers_clean = all_data["sellers"].copy()
sellers_clean = standardize_text_columns(sellers_clean, ["seller_category", "seller_city"])
log_cleaning("sellers", "seller_category/city", "trimmed + standardized casing", len(sellers_clean))

# ---- Refunds ----
refunds_clean = all_data["refunds"].copy()
refunds_clean = standardize_text_columns(refunds_clean, ["refund_status", "refund_reason", "refund_channel"])
refunds_clean = refunds_clean.rename(columns={"refund_status": "refund_status_clean"})
refunds_clean["refund_status_clean"] = refunds_clean["refund_status_clean"].replace({"Not Applicable": "Not Applicable"})
for col in ["refund_initiated_date", "refund_completed_date"]:
    refunds_clean[col] = pd.to_datetime(refunds_clean[col], errors="coerce")
refunds_clean["refund_amount"] = pd.to_numeric(refunds_clean["refund_amount"], errors="coerce")
log_cleaning("refunds", "refund_status/dates/amount", "standardized status casing, coerced dates and amount", len(refunds_clean))

# ---- Complaints ----
complaints_clean = all_data["complaints"].copy()
complaints_clean = standardize_text_columns(complaints_clean, ["complaint_status", "complaint_type", "sentiment_tag"])
complaints_clean = complaints_clean.rename(columns={"complaint_status": "complaint_status_clean"})
complaints_clean["complaint_date"] = pd.to_datetime(complaints_clean["complaint_date"], errors="coerce")
log_cleaning("complaints", "complaint_status/sentiment/date", "standardized casing, coerced date", len(complaints_clean))

# ---- Tickets ----
tickets_clean = all_data["tickets"].copy()
tickets_clean = standardize_text_columns(tickets_clean, ["ticket_status", "assigned_team", "escalation_flag"])
tickets_clean = tickets_clean.rename(columns={"ticket_status": "ticket_status_clean"})
tickets_clean["ticket_created_date"] = pd.to_datetime(tickets_clean["ticket_created_date"], errors="coerce")
tickets_clean["resolution_time_hours"] = pd.to_numeric(tickets_clean["resolution_time_hours"], errors="coerce")
log_cleaning("tickets", "ticket_status/dates/resolution_time_hours", "standardized casing, coerced date and numeric fields", len(tickets_clean))

cleaning_summary = pd.DataFrame(cleaning_log)

add_debug_fix(
    section="Section 7 — Data Cleaning",
    issue_found="Cleaning only lower-cased order_status and ignored dates, amounts, whitespace, and every other dataset (refunds, complaints, tickets, customers, sellers).",
    fix_applied="Added standardize_text_columns() helper reused across all six datasets, safe date parsing with errors='coerce', and numeric coercion for amount/resolution-time fields, while preserving raw columns for auditability.",
    validation_output=f"Produced {len(cleaning_summary)} cleaning actions across orders, customers, sellers, refunds, complaints, and tickets.",
)

display(cleaning_summary)
display(orders_clean.head())


,dataset,field/type,action,affected_records
0,orders,order_status,"trimmed whitespace, standardized casing -> ord...",134
1,orders,date columns,parsed with errors='coerce',1030
2,orders,order_amount,coerced to numeric,13
3,customers,customer_segment/payment,trimmed + standardized casing,1200
4,sellers,seller_category/city,trimmed + standardized casing,1000
5,refunds,refund_status/dates/amount,"standardized status casing, coerced dates and ...",1300
6,complaints,complaint_status/sentiment/date,"standardized casing, coerced date",1500
7,tickets,ticket_status/dates/resolution_time_hours,"standardized casing, coerced date and numeric ...",1500


,order_id,customer_id,seller_id,order_date,order_time,promised_delivery_date,actual_delivery_date,order_amount,payment_method,order_status_clean,delay_reason,delivery_city,device_type,seller_category
0,ORD0000001,CUST000771,SELL0475,2026-05-30,10:51:46,2026-06-03,2026-06-03,57056.16,Cash On Delivery,Delivered,NaN,Delhi,Ios,Electronics
1,ORD0000002,CUST000122,SELL0035,2026-05-05,15:59:45,2026-05-08,2026-05-07,25038.34,Upi,Delivered,NaN,Gurugram,Ios,Grocery
2,ORD0000003,CUST000784,SELL0785,2026-05-11,11:48:01,2026-05-15,NaT,29439.12,Upi,Processing,Warehouse Delay,Lucknow,Android,Grocery
3,ORD0000004,CUST000288,SELL0211,2026-05-23,03:12:27,2026-05-27,2026-05-25,60697.51,Card,Delivered,NaN,Indore,Android,Beauty
4,ORD0000005,CUST001168,SELL0981,2026-06-14,00:30:13,2026-06-19,NaT,56251.50,Upi,Cancelled,Payment Failed,Nagpur,Mobile Web,Beauty


## Section 8: Order Delay Detection

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`

**How to approach:**
- Compare promised and actual delivery dates.
- Handle pending orders that breached promise date.
- Classify delay buckets.
- Summarize delays by bucket and city.

**Expected output format:**
- delay_summary DataFrame.
- city_delay_summary DataFrame.
- orders table with delivery_delay_days and delay_bucket.

**Hint:**
- Use `actual_delivery_date`, not `delivered_date`.

In [9]:
# SECTION 8 — Order delay detection (repaired)

def calculate_delay_days(row):
    """Delay in days. Delivered orders: actual - promised.
    Pending orders past their promised date: REFERENCE_DATE - promised (current pending delay).
    Missing/invalid promised date: NaN."""
    promised = row["promised_delivery_date"]
    actual = row["actual_delivery_date"]

    if pd.isna(promised):
        return np.nan

    if pd.notna(actual):
        return (actual - promised).days

    # No actual delivery date yet -> still pending/unresolved.
    if REFERENCE_DATE > promised:
        return (REFERENCE_DATE - promised).days
    return 0  # not yet due, treated as currently on-time

def classify_delay_bucket(row):
    promised = row["promised_delivery_date"]
    delay_days = row["delivery_delay_days"]

    if pd.isna(promised) or pd.isna(delay_days):
        return "Delivery Date Missing or Invalid"
    if delay_days <= 0:
        return "On Time"
    if delay_days <= 2:
        return "Minor Delay"
    if delay_days <= 5:
        return "Moderate Delay"
    return "Severe Delay"

orders_clean["delivery_delay_days"] = orders_clean.apply(calculate_delay_days, axis=1)
orders_clean["delay_bucket"] = orders_clean.apply(classify_delay_bucket, axis=1)

delay_summary = (
    orders_clean["delay_bucket"]
    .value_counts()
    .rename_axis("delay_bucket")
    .reset_index(name="order_count")
)

city_delay_summary = (
    orders_clean.groupby("delivery_city")
    .agg(
        order_count=("order_id", "count"),
        avg_delay_days=("delivery_delay_days", "mean"),
        severe_delay_count=("delay_bucket", lambda s: (s == "Severe Delay").sum()),
    )
    .reset_index()
    .sort_values("severe_delay_count", ascending=False)
)

add_debug_fix(
    section="Section 8 — Order Delay Detection",
    issue_found="Code used a non-existent 'delivered_date' column, produced a delay_days column that always failed, and never handled pending orders whose promised date had already passed.",
    fix_applied="Implemented calculate_delay_days() using actual_delivery_date with a documented reference date for pending orders, and classify_delay_bucket() with On Time / Minor / Moderate / Severe / Missing-Invalid buckets.",
    validation_output=f"delivery_delay_days populated for {orders_clean['delivery_delay_days'].notna().sum()} of {len(orders_clean)} orders; bucket distribution: {dict(delay_summary.values.tolist())}",
)

print("Delay bucket distribution:")
display(delay_summary)
print("Delay by delivery city (top by severe-delay count):")
display(city_delay_summary.head(10))
print("Order-level delay preview:")
display(orders_clean[["order_id", "promised_delivery_date", "actual_delivery_date",
                       "delivery_delay_days", "delay_bucket"]].head())


Delay bucket distribution:


,delay_bucket,order_count
0,On Time,2227
1,Severe Delay,1590
2,Minor Delay,765
3,Moderate Delay,400
4,Delivery Date Missing or Invalid,18


Delay by delivery city (top by severe-delay count):


,delivery_city,order_count,avg_delay_days,severe_delay_count
9,Kochi,313,10.048077,117
2,Chennai,344,8.863372,114
6,Hyderabad,294,8.938356,114
15,Pune,319,8.006289,110
11,Lucknow,308,8.301948,108
12,Mumbai,319,8.012618,105
7,Indore,309,8.551948,101
0,Ahmedabad,306,7.009804,98
4,Delhi,327,8.073846,98
5,Gurugram,321,7.212500,98


Order-level delay preview:


,order_id,promised_delivery_date,actual_delivery_date,delivery_delay_days,delay_bucket
0,ORD0000001,2026-06-03,2026-06-03,0.0,On Time
1,ORD0000002,2026-05-08,2026-05-07,-1.0,On Time
2,ORD0000003,2026-05-15,NaT,46.0,Severe Delay
3,ORD0000004,2026-05-27,2026-05-25,-2.0,On Time
4,ORD0000005,2026-06-19,NaT,11.0,Severe Delay


## Section 9: Refund Delay and Refund-Priority Analysis

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `refunds.xlsx`
- `orders.csv`

**How to approach:**
- Parse refund dates.
- Calculate refund age.
- Detect pending/failed refunds.
- Detect refund SLA breaches and amount mismatches.

**Expected output format:**
- refund_summary table.
- refunds table with refund_priority_candidate and flags.

**Hint:**
- A pending refund needs age calculated against a reference date.

In [10]:
# SECTION 9 — Refund delay and refund-priority analysis (repaired)

def summarize_refund_risk(refunds_df, orders_df):
    """Compute per-refund risk signals against the cleaned orders table."""
    df = refunds_df.merge(
        orders_df[["order_id", "order_amount", "delay_bucket"]], on="order_id", how="left"
    )

    df["refund_age_days"] = np.where(
        df["refund_completed_date"].isna(),
        (REFERENCE_DATE - df["refund_initiated_date"]).dt.days,
        np.nan,
    )
    df["refund_completion_delay_days"] = (
        df["refund_completed_date"] - df["refund_initiated_date"]
    ).dt.days

    status = df["refund_status_clean"].str.lower()
    df["pending_refund"] = status.eq("pending") | status.eq("initiated")
    df["failed_refund"] = status.eq("failed") | status.eq("rejected")

    df["refund_sla_breach"] = (
        (df["refund_age_days"].fillna(-1) > REFUND_SLA_DAYS)
        | (df["refund_completion_delay_days"].fillna(-1) > REFUND_SLA_DAYS)
    )

    df["refund_amount_mismatch"] = (
        (df["refund_amount"] < 0) | (df["refund_amount"] > df["order_amount"])
    )

    df["delay_linked_refund"] = df["delay_bucket"].isin(
        ["Minor Delay", "Moderate Delay", "Severe Delay"]
    )

    df["refund_priority_candidate"] = (
        (df["pending_refund"] | df["failed_refund"]) & df["delay_linked_refund"]
    )

    return df

refunds_analysis = summarize_refund_risk(refunds_clean, orders_clean)

refund_summary = pd.DataFrame({
    "metric": [
        "total_refund_records", "pending_refunds", "failed_refunds", "refund_sla_breaches",
        "refund_amount_mismatches", "refund_priority_candidates",
    ],
    "value": [
        len(refunds_analysis),
        int(refunds_analysis["pending_refund"].sum()),
        int(refunds_analysis["failed_refund"].sum()),
        int(refunds_analysis["refund_sla_breach"].sum()),
        int(refunds_analysis["refund_amount_mismatch"].sum()),
        int(refunds_analysis["refund_priority_candidate"].sum()),
    ],
})

add_debug_fix(
    section="Section 9 — Refund Delay and Refund-Priority Analysis",
    issue_found="refunds_df was pulled straight from all_data (never loaded correctly before this fix) and referenced non-existent 'completed_date'/'initiated_date' columns; no SLA, pending/failed, or mismatch logic existed.",
    fix_applied="Implemented summarize_refund_risk() using the real refund_initiated_date/refund_completed_date columns, a documented reference date for pending refund age (BR-03), and flags for pending/failed refunds, SLA breach, amount mismatch, and delay-linked refund priority (BR-02).",
    validation_output=f"{int(refunds_analysis['refund_priority_candidate'].sum())} refund-priority candidates identified out of {len(refunds_analysis)} refund records.",
)

display(refund_summary)
display(refunds_analysis[["refund_id", "order_id", "refund_status_clean", "refund_amount",
                           "refund_age_days", "refund_completion_delay_days", "pending_refund",
                           "failed_refund", "refund_sla_breach", "refund_amount_mismatch",
                           "refund_priority_candidate"]].head())


,metric,value
0,total_refund_records,1310
1,pending_refunds,436
2,failed_refunds,221
3,refund_sla_breaches,618
4,refund_amount_mismatches,274
5,refund_priority_candidates,354


,refund_id,order_id,refund_status_clean,refund_amount,refund_age_days,refund_completion_delay_days,pending_refund,failed_refund,refund_sla_breach,refund_amount_mismatch,refund_priority_candidate
0,REF000001,ORD0000498,Rejected,44569.30,NaN,8.0,False,True,True,False,False
1,REF000002,ORD0001547,Initiated,40965.07,19.0,NaN,True,False,True,True,False
2,REF000003,ORD0003579,Processed,14072.25,NaN,7.0,False,False,False,False,False
3,REF000004,ORD0000632,Pending,45492.77,15.0,NaN,True,False,True,False,False
4,REF000005,ORD0000671,Processed,42444.88,NaN,3.0,False,False,False,False,False


## Section 10: Customer Complaint Linking

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `customer_complaints.json`
- `orders.csv`
- `customers.csv`

**How to approach:**
- Aggregate complaints by order_id.
- Count negative or angry complaints.
- Detect repeated complaints.
- Create complaint type summary.

**Expected output format:**
- complaints_order table.
- complaint_type_summary table.

**Hint:**
- Aggregate before merging into the order-level feature table.

In [11]:
# SECTION 10 — Customer complaint linking (repaired)

def aggregate_complaints(complaints_df):
    """Aggregate complaint records to one row per order_id. Must run BEFORE merging into orders."""
    df = complaints_df.copy()
    sentiment = df["sentiment_tag"].str.lower()
    df["is_negative_sentiment"] = sentiment.isin(["negative", "frustrated"])
    df["is_angry_sentiment"] = sentiment.eq("angry")

    grouped = df.groupby("order_id").agg(
        complaint_count=("complaint_id", "count"),
        negative_complaint_count=("is_negative_sentiment", "sum"),
        angry_complaint_count=("is_angry_sentiment", "sum"),
        latest_complaint_date=("complaint_date", "max"),
    ).reset_index()

    grouped["repeated_complaint_flag"] = grouped["complaint_count"] > 1
    return grouped

complaints_by_order = aggregate_complaints(complaints_clean)

complaint_type_summary = (
    complaints_clean.groupby("complaint_type")
    .agg(
        complaint_count=("complaint_id", "count"),
        angry_count=("sentiment_tag", lambda s: (s.str.lower() == "angry").sum()),
    )
    .reset_index()
    .sort_values("complaint_count", ascending=False)
)

repeated_complaint_customers = (
    complaints_clean.groupby("customer_id")["complaint_id"].count()
    .reset_index(name="complaint_count")
    .query("complaint_count > 1")
    .sort_values("complaint_count", ascending=False)
)

add_debug_fix(
    section="Section 10 — Customer Complaint Linking",
    issue_found="complaints_df was read from all_data before it was actually loaded, and the groupby used a non-existent 'order' column instead of 'order_id'; no sentiment or repeated-complaint logic existed.",
    fix_applied="Implemented aggregate_complaints() to group by order_id (using the actual sentiment_tag values: Angry, Negative, Neutral, Positive) and flag repeated complaints per order, plus a separate repeated-complaint-per-customer view.",
    validation_output=f"{len(complaints_by_order)} orders have at least one complaint; {int(complaints_by_order['repeated_complaint_flag'].sum())} orders have repeated complaints; {len(repeated_complaint_customers)} customers have more than one complaint.",
)

print("Complaints aggregated by order (preview):")
display(complaints_by_order.head())
print("Complaint type summary:")
display(complaint_type_summary)


Complaints aggregated by order (preview):


,order_id,complaint_count,negative_complaint_count,angry_complaint_count,latest_complaint_date,repeated_complaint_flag
0,ORD0000001,2,1,0,2026-05-18,True
1,ORD0000006,1,0,0,2026-05-31,False
2,ORD0000007,2,0,1,2026-06-05,True
3,ORD0000018,1,1,0,2026-05-22,False
4,ORD0000024,1,0,1,2026-05-24,False


Complaint type summary:


,complaint_type,complaint_count,angry_count
5,Refund Delay,196,73
0,Cancellation Issue,194,52
1,Damaged Product,194,38
7,Wrong Item,193,36
2,Late Delivery,187,47
4,Payment Deducted Twice,183,37
6,Seller Issue,177,32
3,Missing Item,176,40


## Section 11: Support Ticket Mapping

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `support_tickets.csv`
- `orders.csv`

**How to approach:**
- Standardize ticket status.
- Flag escalations.
- Create SLA breach logic.
- Aggregate ticket signals by order_id.

**Expected output format:**
- tickets_order table.
- ticket_team_summary table.

**Hint:**
- Use `resolution_time_hours`, not `resolution_hours`.

In [12]:
# SECTION 11 — Support ticket mapping (repaired)

def aggregate_support_tickets(tickets_df):
    """Standardize ticket signals and aggregate to one row per order_id."""
    df = tickets_df.copy()
    df["ticket_escalated"] = df["escalation_flag"].str.lower().eq("yes")

    # SLA breach uses the documented threshold (TICKET_SLA_HOURS = 24h).
    # Escalated tickets with missing resolution time are treated as an SLA breach
    # (unresolved + escalated is itself a risk signal), otherwise missing resolution
    # time is treated as "not yet breached".
    df["ticket_sla_breach"] = np.where(
        df["resolution_time_hours"].notna(),
        df["resolution_time_hours"] > TICKET_SLA_HOURS,
        df["ticket_escalated"],
    )

    df["ticket_open"] = df["ticket_status_clean"].str.lower().isin(["open", "pending", "in progress"])

    grouped = df.groupby("order_id").agg(
        ticket_count=("ticket_id", "count"),
        escalated_ticket_count=("ticket_escalated", "sum"),
        ticket_sla_breach_count=("ticket_sla_breach", "sum"),
        open_ticket_count=("ticket_open", "sum"),
        max_resolution_time_hours=("resolution_time_hours", "max"),
    ).reset_index()

    return grouped, df

tickets_order, tickets_enriched = aggregate_support_tickets(tickets_clean)

ticket_team_summary = (
    tickets_enriched.groupby("assigned_team")
    .agg(
        ticket_count=("ticket_id", "count"),
        escalated_count=("ticket_escalated", "sum"),
        sla_breach_count=("ticket_sla_breach", "sum"),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
    )
    .reset_index()
    .sort_values("escalated_count", ascending=False)
)

add_debug_fix(
    section="Section 11 — Support Ticket Mapping",
    issue_found="Code referenced a non-existent 'resolution_hours' column and had no escalation detection or order-level aggregation.",
    fix_applied="Corrected the column to resolution_time_hours, applied the documented 24-hour SLA threshold, handled missing resolution time on escalated tickets, and aggregated ticket signals per order_id.",
    validation_output=f"{tickets_order['ticket_count'].sum()} tickets aggregated across {len(tickets_order)} orders; {int(tickets_order['escalated_ticket_count'].sum())} escalated, {int(tickets_order['ticket_sla_breach_count'].sum())} SLA breaches.",
)

print("Tickets aggregated by order (preview):")
display(tickets_order.head())
print("Ticket summary by assigned team:")
display(ticket_team_summary)


Tickets aggregated by order (preview):


,order_id,ticket_count,escalated_ticket_count,ticket_sla_breach_count,open_ticket_count,max_resolution_time_hours
0,ORD0000005,3,1,2,1,114.0
1,ORD0000012,1,1,1,1,31.0
2,ORD0000014,2,0,2,1,168.0
3,ORD0000016,1,0,1,0,60.0
4,ORD0000021,1,0,1,0,50.0


Ticket summary by assigned team:


,assigned_team,ticket_count,escalated_count,sla_breach_count,avg_resolution_time_hours
3,Refund Ops,264,57,218,95.914530
2,Logistics Ops,253,54,211,94.367257
5,Support Ops,234,52,189,89.819905
0,City Ops,260,52,213,90.722222
4,Seller Ops,267,47,222,92.561983
1,Escalation Desk,222,42,178,87.381188


## Section 12: Duplicate and Repeat Issue Detection

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`
- `customer_complaints.json`
- `support_tickets.csv`

**How to approach:**
- Find duplicate order IDs.
- Find duplicate-looking order patterns.
- Find customers with repeated complaints.
- Summarize issue counts.

**Expected output format:**
- duplicate_summary table.
- Optional preview of repeated complaint customers.

**Hint:**
- Duplicate-looking records may share customer, seller, date, and amount even if row IDs differ.

In [13]:
# SECTION 12 — Duplicate and repeat issue detection (repaired)

duplicate_order_ids = int(orders_clean["order_id"].duplicated().sum())

duplicate_looking_orders = int(
    orders_clean.duplicated(subset=["customer_id", "seller_id", "order_date", "order_amount"]).sum()
)

customers_with_repeated_complaints = int((complaints_by_order["complaint_count"] > 1).sum())

orders_with_repeated_tickets = int((tickets_order["ticket_count"] > 1).sum())

duplicate_summary = pd.DataFrame({
    "check": [
        "duplicate_order_ids",
        "duplicate_looking_orders (same customer/seller/date/amount)",
        "orders_with_repeated_complaints",
        "orders_with_repeated_tickets",
    ],
    "count": [
        duplicate_order_ids,
        duplicate_looking_orders,
        customers_with_repeated_complaints,
        orders_with_repeated_tickets,
    ],
})

repeated_complaint_preview = repeated_complaint_customers.head(10)

add_debug_fix(
    section="Section 12 — Duplicate and Repeat Issue Detection",
    issue_found="Only raw fully-duplicated rows were checked; duplicate-looking orders (same customer/seller/date/amount with different IDs), repeated complaints, and repeated ticket issues were never detected.",
    fix_applied="Added checks for duplicate order_id values, near-duplicate order patterns on key business fields, customers with repeated complaints, and orders with more than one support ticket. No records were deleted.",
    validation_output=f"{duplicate_order_ids} duplicate order_id values and {duplicate_looking_orders} duplicate-looking order records flagged for review.",
)

display(duplicate_summary)
print("Customers with repeated complaints (preview):")
display(repeated_complaint_preview)


,check,count
0,duplicate_order_ids,16
1,duplicate_looking_orders (same customer/seller...,0
2,orders_with_repeated_complaints,183
3,orders_with_repeated_tickets,197


Customers with repeated complaints (preview):


,customer_id,complaint_count
235,CUST000364,7
651,CUST000973,6
561,CUST000829,6
443,CUST000664,6
118,CUST000179,5
737,CUST001098,5
56,CUST000079,5
697,CUST001038,5
206,CUST000321,5
294,CUST000447,5


## Section 13: Final Feature Table / Master Merge

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`
- `customers.csv`
- `sellers.csv`
- `refunds.xlsx`
- `customer_complaints.json`
- `support_tickets.csv`

**How to approach:**
- Aggregate many-to-one tables first.
- Merge customer and seller master data.
- Merge aggregated refund/complaint/ticket features.
- Validate row count before and after merge.

**Expected output format:**
- final_feature_table DataFrame.
- merge_validation table.

**Hint:**
- Raw refund, complaint, and ticket merges can multiply rows if not aggregated first.

In [14]:
# SECTION 13 — Final feature table / master merge (repaired)

rows_before_merge = len(orders_clean)

# Aggregate refund signals to one row per order BEFORE merging (avoids row explosion).
refund_agg = refunds_analysis.groupby("order_id").agg(
    refund_count=("refund_id", "count"),
    pending_refund_count=("pending_refund", "sum"),
    failed_refund_count=("failed_refund", "sum"),
    refund_sla_breach_count=("refund_sla_breach", "sum"),
    refund_amount_mismatch_count=("refund_amount_mismatch", "sum"),
    refund_priority_candidate_count=("refund_priority_candidate", "sum"),
).reset_index()

final_feature_table = orders_clean.copy()
r1 = len(final_feature_table)

final_feature_table = final_feature_table.merge(
    customers_clean[["customer_id", "customer_segment", "account_age_days", "total_orders"]],
    on="customer_id", how="left"
)
r2 = len(final_feature_table)

final_feature_table = final_feature_table.merge(
    sellers_clean[["seller_id", "seller_rating", "seller_city"]],
    on="seller_id", how="left"
)
r3 = len(final_feature_table)

final_feature_table = final_feature_table.merge(refund_agg, on="order_id", how="left")
r4 = len(final_feature_table)

final_feature_table = final_feature_table.merge(complaints_by_order, on="order_id", how="left")
r5 = len(final_feature_table)

final_feature_table = final_feature_table.merge(tickets_order, on="order_id", how="left")
r6 = len(final_feature_table)

# Orders without refunds/complaints/tickets should show as zero, not NaN.
zero_fill_cols = [
    "refund_count", "pending_refund_count", "failed_refund_count", "refund_sla_breach_count",
    "refund_amount_mismatch_count", "refund_priority_candidate_count",
    "complaint_count", "negative_complaint_count", "angry_complaint_count",
    "ticket_count", "escalated_ticket_count", "ticket_sla_breach_count", "open_ticket_count",
]
for col in zero_fill_cols:
    final_feature_table[col] = final_feature_table[col].fillna(0).astype(int)
final_feature_table["repeated_complaint_flag"] = final_feature_table["repeated_complaint_flag"].fillna(False)

merge_validation = pd.DataFrame({
    "stage": [
        "rows_before_merge", "rows_after_customer_merge", "rows_after_seller_merge",
        "rows_after_refund_merge", "rows_after_complaint_merge", "rows_after_ticket_merge",
    ],
    "row_count": [rows_before_merge, r2, r3, r4, r5, r6],
})

add_debug_fix(
    section="Section 13 — Final Feature Table / Master Merge",
    issue_found="Raw refund, complaint, and ticket tables were merged directly onto orders without aggregating first, which multiplies rows for orders with more than one refund/complaint/ticket.",
    fix_applied="Aggregated refunds, complaints, and tickets to one row per order_id first, then merged customer, seller, and aggregate features onto orders_clean with left joins, filling zeros for orders with no related records.",
    validation_output=f"Row count stayed at {rows_before_merge} through every merge step (no accidental row multiplication): {merge_validation['row_count'].tolist()}",
)

display(merge_validation)
print("Final feature table shape:", final_feature_table.shape)
display(final_feature_table[["order_id", "customer_id", "seller_id", "delivery_city", "order_amount",
                              "order_status_clean", "delivery_delay_days", "delay_bucket",
                              "refund_count", "complaint_count", "ticket_count"]].head())


,stage,row_count
0,rows_before_merge,5000
1,rows_after_customer_merge,5000
2,rows_after_seller_merge,5000
3,rows_after_refund_merge,5000
4,rows_after_complaint_merge,5000
5,rows_after_ticket_merge,5000


Final feature table shape: (5000, 37)


,order_id,customer_id,seller_id,delivery_city,order_amount,order_status_clean,delivery_delay_days,delay_bucket,refund_count,complaint_count,ticket_count
0,ORD0000001,CUST000771,SELL0475,Delhi,57056.16,Delivered,0.0,On Time,0,2,0
1,ORD0000002,CUST000122,SELL0035,Gurugram,25038.34,Delivered,-1.0,On Time,1,0,0
2,ORD0000003,CUST000784,SELL0785,Lucknow,29439.12,Processing,46.0,Severe Delay,0,0,0
3,ORD0000004,CUST000288,SELL0211,Indore,60697.51,Delivered,-2.0,On Time,0,0,0
4,ORD0000005,CUST001168,SELL0981,Nagpur,56251.50,Cancelled,11.0,Severe Delay,1,0,3


## Section 14: Priority Classification

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Create a scoring function.
- Combine delay, refund, complaint, ticket, and customer segment signals.
- Return priority level and reasons.

**Expected output format:**
- priority_summary table.
- final_feature_table with issue_score, issue_priority, priority_reasons.

**Hint:**
- Priority should not be based on delay alone.

In [15]:
# SECTION 14 — Priority classification (repaired)

def classify_issue_priority(row):
    """Transparent, additive rule-based score combining multiple business signals (BR-08)."""
    score = 0
    reasons = []

    if row["delay_bucket"] == "Severe Delay":
        score += 30; reasons.append("Severe delivery delay")
    elif row["delay_bucket"] == "Moderate Delay":
        score += 15; reasons.append("Moderate delivery delay")
    elif row["delay_bucket"] == "Minor Delay":
        score += 5; reasons.append("Minor delivery delay")
    elif row["delay_bucket"] == "Delivery Date Missing or Invalid":
        score += 5; reasons.append("Delivery date missing or invalid")

    if row["pending_refund_count"] > 0:
        score += 15; reasons.append("Pending refund")
    if row["failed_refund_count"] > 0:
        score += 20; reasons.append("Failed refund")
    if row["refund_sla_breach_count"] > 0:
        score += 15; reasons.append("Refund SLA breach")
    if row["refund_amount_mismatch_count"] > 0:
        score += 10; reasons.append("Refund amount mismatch")

    if row["angry_complaint_count"] > 0:
        score += 15; reasons.append("Angry customer complaint")
    elif row["negative_complaint_count"] > 0:
        score += 10; reasons.append("Negative customer complaint")
    if row["repeated_complaint_flag"]:
        score += 10; reasons.append("Repeated complaints on this order")

    if row["escalated_ticket_count"] > 0:
        score += 20; reasons.append("Escalated support ticket")
    if row["ticket_sla_breach_count"] > 0:
        score += 15; reasons.append("Support ticket SLA breach")

    if score > 0 and row.get("customer_segment") in ("Premium", "Business"):
        score += 5; reasons.append(f"High-value customer segment ({row['customer_segment']})")

    if score >= 70:
        priority = "Critical"
    elif score >= 40:
        priority = "High"
    elif score >= 15:
        priority = "Medium"
    else:
        priority = "Low"

    return pd.Series({
        "issue_score": score,
        "issue_priority": priority,
        "priority_reasons": "; ".join(reasons) if reasons else "No significant risk signals detected",
    })

priority_results = final_feature_table.apply(classify_issue_priority, axis=1)
final_feature_table = pd.concat([final_feature_table, priority_results], axis=1)

priority_summary = (
    final_feature_table["issue_priority"]
    .value_counts()
    .reindex(["Critical", "High", "Medium", "Low"])
    .fillna(0)
    .astype(int)
    .rename_axis("issue_priority")
    .reset_index(name="order_count")
)

add_debug_fix(
    section="Section 14 — Priority Classification",
    issue_found="classify_priority() only checked one non-existent column ('delay_days') and returned High/Low with no other signals — violating BR-08 (priority must combine multiple signals).",
    fix_applied="Implemented classify_issue_priority() as a transparent additive scoring function combining delay severity, refund, complaint, ticket, and customer-segment signals, producing issue_score, issue_priority (Critical/High/Medium/Low), and priority_reasons.",
    validation_output=f"Priority distribution: {dict(zip(priority_summary['issue_priority'], priority_summary['order_count']))}",
)

display(priority_summary)
display(final_feature_table[["order_id", "issue_score", "issue_priority", "priority_reasons"]].head())


,issue_priority,order_count
0,Critical,149
1,High,903
2,Medium,2199
3,Low,1749


,order_id,issue_score,issue_priority,priority_reasons
0,ORD0000001,20,Medium,Negative customer complaint; Repeated complain...
1,ORD0000002,30,Medium,Pending refund; Refund SLA breach
2,ORD0000003,30,Medium,Severe delivery delay
3,ORD0000004,0,Low,No significant risk signals detected
4,ORD0000005,95,Critical,Severe delivery delay; Pending refund; Refund ...


## Section 15: Recommended Action Generator

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Create a function that reads priority and reasons.
- Return specific support/refund/seller/city actions.
- Add recommended_action to final table.

**Expected output format:**
- Action summary table.
- Preview with order_id, priority, action.

**Hint:**
- Recommendations should help a support lead decide what to do next.

In [16]:
# SECTION 15 — Recommended action generator (repaired)

def generate_recommended_action(row):
    """Build a specific, multi-signal recommended action from the actual case signals."""
    actions = []

    if row["failed_refund_count"] > 0 and row["refund_amount_mismatch_count"] > 0:
        actions.append("Refund reconciliation: investigate failed refund with amount mismatch")
    elif row["pending_refund_count"] > 0 and row["refund_sla_breach_count"] > 0:
        actions.append("Refund operations follow-up: pending refund has breached SLA")
    elif row["pending_refund_count"] > 0:
        actions.append("Refund operations review: refund still pending")
    elif row["failed_refund_count"] > 0:
        actions.append("Refund operations review: failed refund needs retry/investigation")

    if row["escalated_ticket_count"] > 0 and row["angry_complaint_count"] > 0:
        actions.append("Escalation desk review: escalated ticket with angry customer complaint")
    elif row["escalated_ticket_count"] > 0:
        actions.append("Escalation desk review: open escalated support ticket")
    elif row["ticket_sla_breach_count"] > 0:
        actions.append("Support follow-up: ticket resolution time has breached SLA")

    if row["delay_bucket"] == "Severe Delay":
        actions.append("Delivery investigation: severe delivery delay needs logistics review")
    elif row["delay_bucket"] == "Moderate Delay":
        actions.append("Delivery investigation: moderate delivery delay, monitor courier/seller dispatch")

    if row["angry_complaint_count"] > 0 or row["repeated_complaint_flag"]:
        actions.append("Customer communication: proactively update customer given complaint history")

    if row["issue_priority"] in ("Critical", "High") and row["delay_bucket"] in ("Moderate Delay", "Severe Delay"):
        actions.append(f"Seller operations review: recurring delay pattern for seller {row['seller_id']}")

    if not actions:
        if row["issue_priority"] == "Medium":
            actions.append("Support follow-up: monitor order, no urgent action required yet")
        else:
            actions.append("No action needed: order shows no significant risk signals")

    return " | ".join(dict.fromkeys(actions))  # de-duplicate while preserving order

final_feature_table["recommended_action"] = final_feature_table.apply(generate_recommended_action, axis=1)

action_summary = (
    final_feature_table["recommended_action"]
    .value_counts()
    .head(10)
    .rename_axis("recommended_action")
    .reset_index(name="order_count")
)

add_debug_fix(
    section="Section 15 — Recommended Action Generator",
    issue_found="generate_action() (called recommend_action) returned the same generic string 'Review order' for every case regardless of signals.",
    fix_applied="Implemented generate_recommended_action() to build a specific, combined action string from refund, ticket, delay, complaint, and seller-pattern signals for each order.",
    validation_output=f"{final_feature_table['recommended_action'].nunique()} distinct recommended-action combinations generated across {len(final_feature_table)} orders.",
)

display(action_summary)
display(final_feature_table[["order_id", "issue_priority", "recommended_action"]].head())


,recommended_action,order_count
0,No action needed: order shows no significant r...,1742
1,Delivery investigation: severe delivery delay ...,891
2,Support follow-up: ticket resolution time has ...,389
3,Delivery investigation: moderate delivery dela...,248
4,Customer communication: proactively update cus...,178
5,Refund operations follow-up: pending refund ha...,173
6,Escalation desk review: open escalated support...,141
7,"Support follow-up: monitor order, no urgent ac...",127
8,Refund operations review: failed refund needs ...,60
9,Support follow-up: ticket resolution time has ...,51


,order_id,issue_priority,recommended_action
0,ORD0000001,Medium,Customer communication: proactively update cus...
1,ORD0000002,Medium,Refund operations follow-up: pending refund ha...
2,ORD0000003,Medium,Delivery investigation: severe delivery delay ...
3,ORD0000004,Low,No action needed: order shows no significant r...
4,ORD0000005,Critical,Refund operations follow-up: pending refund ha...


## Section 16: AI-Ready Prompt Generator

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Create structured prompts from case data.
- Include only available facts.
- Do not call external APIs.
- Add prompt preview column.

**Expected output format:**
- Preview table with order_id and ai_prompt_preview.

**Hint:**
- This is prompt generation only, not AI integration.

In [17]:
# SECTION 16 — AI-ready prompt generator (repaired)

def generate_ai_prompt(row):
    """Build a structured, fact-only prompt for a future approved support assistant.
    This function only generates text — it never calls an external AI API."""
    prompt = (
        f"Order {row['order_id']} (customer {row['customer_id']}, seller {row['seller_id']}, "
        f"city {row['delivery_city']}, amount Rs.{row['order_amount']}, status {row['order_status_clean']}).\n"
        f"Delivery: {row['delay_bucket']} ({row['delivery_delay_days']} day(s) delay).\n"
        f"Refunds: {row['refund_count']} total, {row['pending_refund_count']} pending, "
        f"{row['failed_refund_count']} failed, {row['refund_sla_breach_count']} SLA breach(es), "
        f"{row['refund_amount_mismatch_count']} amount mismatch(es).\n"
        f"Complaints: {row['complaint_count']} total, {row['angry_complaint_count']} angry, "
        f"{row['negative_complaint_count']} negative, repeated={row['repeated_complaint_flag']}.\n"
        f"Support tickets: {row['ticket_count']} total, {row['escalated_ticket_count']} escalated, "
        f"{row['ticket_sla_breach_count']} SLA breach(es).\n"
        f"Priority: {row['issue_priority']} (score {row['issue_score']}). Reasons: {row['priority_reasons']}.\n"
        f"Recommended action: {row['recommended_action']}.\n"
        "Using only the facts above, draft a short, appropriate support/operations response or next step for this case."
    )
    return prompt

final_feature_table["ai_prompt_preview"] = final_feature_table.apply(generate_ai_prompt, axis=1)

add_debug_fix(
    section="Section 16 — AI-Ready Prompt Generator",
    issue_found="create_ai_prompt() returned a fixed placeholder string ('Write a message to customer') with no order, delay, refund, complaint, ticket, or priority context.",
    fix_applied="Implemented generate_ai_prompt() to assemble a fact-only structured prompt per order from the final feature table columns. No external AI API is called anywhere in the notebook.",
    validation_output=f"ai_prompt_preview populated for all {final_feature_table['ai_prompt_preview'].notna().sum()} orders.",
)

preview = final_feature_table[["order_id", "ai_prompt_preview"]].head(3)
for _, r in preview.iterrows():
    print(f"--- {r['order_id']} ---")
    print(r["ai_prompt_preview"])
    print()


--- ORD0000001 ---
Order ORD0000001 (customer CUST000771, seller SELL0475, city Delhi, amount Rs.57056.16, status Delivered).
Delivery: On Time (0.0 day(s) delay).
Refunds: 0 total, 0 pending, 0 failed, 0 SLA breach(es), 0 amount mismatch(es).
Complaints: 2 total, 0 angry, 1 negative, repeated=True.
Support tickets: 0 total, 0 escalated, 0 SLA breach(es).
Priority: Medium (score 20). Reasons: Negative customer complaint; Repeated complaints on this order.
Recommended action: Customer communication: proactively update customer given complaint history.
Using only the facts above, draft a short, appropriate support/operations response or next step for this case.

--- ORD0000002 ---
Order ORD0000002 (customer CUST000122, seller SELL0035, city Gurugram, amount Rs.25038.34, status Delivered).
Delivery: On Time (-1.0 day(s) delay).
Refunds: 1 total, 1 pending, 0 failed, 1 SLA breach(es), 0 amount mismatch(es).
Complaints: 0 total, 0 angry, 0 negative, repeated=False.
Support tickets: 0 total,

## Section 17: Colab GUI/Search/Filter Experience

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Use ipywidgets or equivalent.
- Allow filtering by priority, city, and ID search.
- Display a compact case table.

**Expected output format:**
- Interactive widget or fallback output.
- Filtered case preview.

**Hint:**
- The GUI should make the notebook feel like an internal tool.

In [18]:
# SECTION 17 — Colab GUI/search/filter experience (repaired)

DISPLAY_COLUMNS = ["order_id", "customer_id", "delivery_city", "issue_priority", "issue_score",
                    "delay_bucket", "pending_refund_count", "failed_refund_count",
                    "escalated_ticket_count", "recommended_action"]

def filter_cases(priority="All", city="All", search_text=""):
    df = final_feature_table
    if priority != "All":
        df = df[df["issue_priority"] == priority]
    if city != "All":
        df = df[df["delivery_city"] == city]
    if search_text:
        text = search_text.strip().lower()
        mask = (
            df["order_id"].str.lower().str.contains(text, na=False)
            | df["customer_id"].str.lower().str.contains(text, na=False)
        )
        df = df[mask]
    return df[DISPLAY_COLUMNS].head(25)

try:
    import ipywidgets as widgets
    from IPython.display import display as ipy_display, clear_output

    priority_options = ["All"] + sorted(final_feature_table["issue_priority"].dropna().unique().tolist())
    city_options = ["All"] + sorted(final_feature_table["delivery_city"].dropna().unique().tolist())

    priority_dropdown = widgets.Dropdown(options=priority_options, description="Priority")
    city_dropdown = widgets.Dropdown(options=city_options, description="City")
    search_box = widgets.Text(description="Search")
    output_area = widgets.Output()

    def on_change(change=None):
        with output_area:
            clear_output(wait=True)
            result = filter_cases(priority_dropdown.value, city_dropdown.value, search_box.value)
            print(f"{len(result)} matching case(s) shown (max 25):")
            ipy_display(result)

    priority_dropdown.observe(on_change, names="value")
    city_dropdown.observe(on_change, names="value")
    search_box.observe(on_change, names="value")

    ipy_display(widgets.HBox([priority_dropdown, city_dropdown, search_box]))
    ipy_display(output_area)
    on_change()

except Exception as exc:
    print("ipywidgets unavailable, falling back to a static filtered preview:", exc)
    display(filter_cases(priority="High"))


Output()

## Section 18: Final Report Export and Preview

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Select required report schema columns.
- Add generated timestamp.
- Export final CSV.
- Preview top priority cases.

**Expected output format:**
- final_issue_resolution_report.csv.
- Priority summary table.
- Final report preview.

**Hint:**
- The final report should include business signals, not just IDs.

In [19]:
# SECTION 18 — Final report export and preview (repaired)

REPORT_SCHEMA = [
    "order_id", "customer_id", "seller_id", "delivery_city", "order_amount",
    "order_status_clean", "delivery_delay_days", "delay_bucket",
    "refund_count", "pending_refund_count", "failed_refund_count",
    "complaint_count", "ticket_count", "issue_score", "issue_priority",
    "recommended_action", "ai_prompt_preview",
]

def export_final_report(feature_table, schema, output_path):
    report = feature_table[schema].copy()
    report["report_generated_at"] = REFERENCE_DATE.strftime("%Y-%m-%d")
    report.to_csv(output_path, index=False)
    return report

REPORT_PATH = "final_issue_resolution_report.csv"
final_report = export_final_report(final_feature_table, REPORT_SCHEMA, REPORT_PATH)

add_debug_fix(
    section="Section 18 — Final Report Export and Preview",
    issue_found="final_report only kept order_id and issue_priority, and no CSV was ever exported.",
    fix_applied="Implemented export_final_report() to select the full required PRD schema, add a generated timestamp, and write final_issue_resolution_report.csv.",
    validation_output=f"Exported {REPORT_PATH} with shape {final_report.shape}.",
)

print("Report exported to:", REPORT_PATH)
print("Report shape:", final_report.shape)
print("Report schema:", list(final_report.columns))
print()
print("Priority summary:")
display(priority_summary)
print("Highest-priority case preview:")
display(final_report.sort_values("issue_score", ascending=False).head(5))


Report exported to: final_issue_resolution_report.csv
Report shape: (5000, 18)
Report schema: ['order_id', 'customer_id', 'seller_id', 'delivery_city', 'order_amount', 'order_status_clean', 'delivery_delay_days', 'delay_bucket', 'refund_count', 'pending_refund_count', 'failed_refund_count', 'complaint_count', 'ticket_count', 'issue_score', 'issue_priority', 'recommended_action', 'ai_prompt_preview', 'report_generated_at']

Priority summary:


,issue_priority,order_count
0,Critical,149
1,High,903
2,Medium,2199
3,Low,1749


Highest-priority case preview:


,order_id,customer_id,seller_id,delivery_city,order_amount,order_status_clean,delivery_delay_days,delay_bucket,refund_count,pending_refund_count,failed_refund_count,complaint_count,ticket_count,issue_score,issue_priority,recommended_action,ai_prompt_preview,report_generated_at
3695,ORD0003696,CUST000395,SELL0307,Chennai,59530.80,Processing,13.0,Severe Delay,3,1,1,0,1,125,Critical,Refund reconciliation: investigate failed refu...,"Order ORD0003696 (customer CUST000395, seller ...",2026-06-30
2686,ORD0002687,CUST000018,SELL0553,Kochi,23047.07,Processing,52.0,Severe Delay,2,1,0,2,1,120,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0002687 (customer CUST000018, seller ...",2026-06-30
749,ORD0000750,CUST000431,SELL0567,Mumbai,40173.12,Delayed,9.0,Severe Delay,2,1,1,2,1,115,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0000750 (customer CUST000431, seller ...",2026-06-30
421,ORD0000422,CUST000756,SELL0957,Mumbai,46937.94,Processing,16.0,Severe Delay,2,1,0,0,1,110,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0000422 (customer CUST000756, seller ...",2026-06-30
1096,ORD0001097,CUST000173,SELL0770,Ahmedabad,50010.47,Returned,9.0,Severe Delay,1,0,1,0,1,100,Critical,Refund operations review: failed refund needs ...,"Order ORD0001097 (customer CUST000173, seller ...",2026-06-30


## Section 19: Final Validation Checks

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_report`
- `debug_log`
- `notebook outputs`

**How to approach:**
- Check report file exists.
- Check required columns.
- Check priorities/actions/prompts are non-null.
- Check debug log evidence exists.

**Expected output format:**
- final_validation_df with check, passed, details.
- Final success message.

**Hint:**
- Validation output is the final engineering proof before handover.

In [20]:
# SECTION 19 — Final validation checks (repaired)

checks = []

def add_check(name, passed, details):
    checks.append({"check": name, "passed": bool(passed), "details": details})

report_exists = Path(REPORT_PATH).exists()
add_check("final_report_file_exists", report_exists, f"Path checked: {REPORT_PATH}")

add_check("final_report_non_empty", len(final_report) > 0, f"Row count: {len(final_report)}")

missing_schema_cols = [c for c in REPORT_SCHEMA if c not in final_report.columns]
add_check("required_columns_present", len(missing_schema_cols) == 0, f"Missing columns: {missing_schema_cols}")

valid_priorities = {"Critical", "High", "Medium", "Low"}
bad_priorities = set(final_report["issue_priority"].unique()) - valid_priorities
add_check("priority_values_valid", len(bad_priorities) == 0, f"Unexpected priority values: {bad_priorities}")

add_check("priorities_not_null", final_report["issue_priority"].isna().sum() == 0,
          f"Null priorities: {int(final_report['issue_priority'].isna().sum())}")

add_check("recommended_actions_not_null", final_report["recommended_action"].isna().sum() == 0,
          f"Null actions: {int(final_report['recommended_action'].isna().sum())}")

add_check("ai_prompts_not_null", final_report["ai_prompt_preview"].isna().sum() == 0,
          f"Null prompts: {int(final_report['ai_prompt_preview'].isna().sum())}")

add_check("issue_score_populated", final_report["issue_score"].notna().all(),
          f"Non-null issue_score count: {int(final_report['issue_score'].notna().sum())} of {len(final_report)}")

add_check("row_count_sensible", len(final_report) == len(orders_clean),
          f"Report rows ({len(final_report)}) match cleaned orders rows ({len(orders_clean)})")

add_check("debug_log_has_evidence", len(debug_log) >= 10, f"Debug log entries: {len(debug_log)}")

major_outputs_exist = all(name in globals() for name in [
    "dataset_load_summary", "validation_summary", "cleaning_summary", "delay_summary",
    "refund_summary", "complaints_by_order", "tickets_order", "duplicate_summary",
    "merge_validation", "priority_summary",
])
add_check("major_outputs_exist", major_outputs_exist, "Checked that key summary objects from every section exist in memory.")

try:
    reread = pd.read_csv(REPORT_PATH)
    readback_ok = len(reread) == len(final_report)
except Exception as exc:
    readback_ok = False
    reread = None
add_check("final_report_readable_back", readback_ok,
          f"Re-read {len(reread) if reread is not None else 0} rows from disk vs {len(final_report)} expected.")

final_validation_df = pd.DataFrame(checks)

add_debug_fix(
    section="Section 19 — Final Validation Checks",
    issue_found="No validation checks existed; the section only printed a placeholder message.",
    fix_applied="Implemented 12 concrete checks covering report existence, schema completeness, null handling, row-count sanity, debug-log evidence, and a read-back test.",
    validation_output=f"{int(final_validation_df['passed'].sum())} of {len(final_validation_df)} checks passed.",
)

display(final_validation_df)

if final_validation_df["passed"].all():
    print("SUCCESS: All final validation checks passed. The notebook is ready for handover.")
else:
    failed = final_validation_df[~final_validation_df["passed"]]["check"].tolist()
    print("INCOMPLETE: The following checks failed and must be fixed before handover:", failed)


,check,passed,details
0,final_report_file_exists,True,Path checked: final_issue_resolution_report.csv
1,final_report_non_empty,True,Row count: 5000
2,required_columns_present,True,Missing columns: []
3,priority_values_valid,True,Unexpected priority values: set()
4,priorities_not_null,True,Null priorities: 0
5,recommended_actions_not_null,True,Null actions: 0
6,ai_prompts_not_null,True,Null prompts: 0
7,issue_score_populated,True,Non-null issue_score count: 5000 of 5000
8,row_count_sensible,True,Report rows (5000) match cleaned orders rows (...
9,debug_log_has_evidence,True,Debug log entries: 14


SUCCESS: All final validation checks passed. The notebook is ready for handover.


## Notebook-Based Deliverable Placeholders

Fill the following sections inside this same notebook. Do not submit these as separate files.

In [21]:
# DELIVERABLE — Debug Fix Log
debug_fix_log = pd.DataFrame(debug_log, columns=["section", "issue_found", "fix_applied", "validation_output"])
display(debug_fix_log)


,section,issue_found,fix_applied,validation_output
0,Section 3 — Basic Order Summary,Code referenced a non-existent 'status' column...,Rebuilt order_summary and order_status_summary...,order_summary has 7 metrics; order_status_summ...
1,Section 5 — Multi-File DataLoader,"Loader used wrong filenames ('customer.csv', '...",Corrected filenames to customers.csv/sellers.c...,All 7 files reported status 'Loaded': True
2,Section 6 — Data Validation,"Validation checked a hard-coded, wrong column ...",Rebuilt validation using real column names and...,"26 checks run; 12 flagged for review, 14 passed."
3,Section 7 — Data Cleaning,Cleaning only lower-cased order_status and ign...,Added standardize_text_columns() helper reused...,"Produced 8 cleaning actions across orders, cus..."
4,Section 8 — Order Delay Detection,Code used a non-existent 'delivered_date' colu...,Implemented calculate_delay_days() using actua...,delivery_delay_days populated for 4982 of 5000...
5,Section 9 — Refund Delay and Refund-Priority A...,refunds_df was pulled straight from all_data (...,Implemented summarize_refund_risk() using the ...,354 refund-priority candidates identified out ...
6,Section 10 — Customer Complaint Linking,complaints_df was read from all_data before it...,Implemented aggregate_complaints() to group by...,1299 orders have at least one complaint; 183 o...
7,Section 11 — Support Ticket Mapping,Code referenced a non-existent 'resolution_hou...,"Corrected the column to resolution_time_hours,...",1500 tickets aggregated across 1287 orders; 30...
8,Section 12 — Duplicate and Repeat Issue Detection,Only raw fully-duplicated rows were checked; d...,"Added checks for duplicate order_id values, ne...",16 duplicate order_id values and 0 duplicate-l...
9,Section 13 — Final Feature Table / Master Merge,"Raw refund, complaint, and ticket tables were ...","Aggregated refunds, complaints, and tickets to...",Row count stayed at 5000 through every merge s...


## AI Prompt Usage Log

Document how AI helped you, if used. You must still validate and explain all final code and outputs yourself.

In [22]:
# DELIVERABLE — AI Prompt Usage Log
# Documents how AI assistance was used while repairing this notebook. No external AI API is called
# anywhere in the notebook itself — generate_ai_prompt() only produces prompt text (Section 16).

ai_prompt_usage_log = pd.DataFrame([
    {
        "task": "Diagnose broken inherited code section by section",
        "prompt_used": "Given the PRD, meeting transcript, and this broken cell, identify the wrong column names/logic and explain the root cause.",
        "ai_output_used": "Used to identify specific bugs (e.g. delivered_date vs actual_delivery_date, resolution_hours vs resolution_time_hours, wrong filenames).",
        "human_validation_done": "Yes — each fix was checked against the real dataset columns and PRD business rules before accepting it.",
    },
    {
        "task": "Design the multi-signal issue_score formula",
        "prompt_used": "Suggest a transparent, additive scoring approach that combines delay, refund, complaint, and ticket signals per BR-08.",
        "ai_output_used": "Used the suggested weighting structure as a starting point for classify_issue_priority().",
        "human_validation_done": "Yes — thresholds and point values were reviewed, documented in code comments, and checked against priority_summary output.",
    },
    {
        "task": "Draft the AI-ready prompt template",
        "prompt_used": "Draft a fact-only prompt template covering order, delay, refund, complaint, ticket, priority, and action fields.",
        "ai_output_used": "Used as the base text template for generate_ai_prompt(); no external AI API is invoked by this function.",
        "human_validation_done": "Yes — verified the template only references columns that exist in final_feature_table.",
    },
])
display(ai_prompt_usage_log)


,task,prompt_used,ai_output_used,human_validation_done
0,Diagnose broken inherited code section by section,"Given the PRD, meeting transcript, and this br...",Used to identify specific bugs (e.g. delivered...,Yes — each fix was checked against the real da...
1,Design the multi-signal issue_score formula,"Suggest a transparent, additive scoring approa...",Used the suggested weighting structure as a st...,Yes — thresholds and point values were reviewe...
2,Draft the AI-ready prompt template,Draft a fact-only prompt template covering ord...,Used as the base text template for generate_ai...,Yes — verified the template only references co...


## PRD Completion Mapping

Map product requirements to notebook sections and evidence.

In [23]:
# DELIVERABLE — PRD Completion Mapping

prd_completion_mapping = pd.DataFrame([
    {"prd_requirement": "FR-01 Load dataset ZIP in Colab", "notebook_section": "Dataset Upload Instructions (pre-Section 1)", "evidence": "ZIP extraction cell creates DATA_DIR and lists extracted files.", "completion_status": "Complete"},
    {"prd_requirement": "FR-02 Load all data files", "notebook_section": "Section 5 — Multi-File DataLoader", "evidence": "FlipkartIssueDataLoader.load_all() loads CSV/Excel/JSON/TXT into all_data; dataset_load_summary shows status=Loaded for all 7 files.", "completion_status": "Complete"},
    {"prd_requirement": "FR-03 Validate data integrity", "notebook_section": "Section 6 — Data Validation", "evidence": "validation_summary reports required columns, duplicate IDs, orphan relationships, invalid dates, and amount anomalies.", "completion_status": "Complete"},
    {"prd_requirement": "FR-04 Clean and standardize data", "notebook_section": "Section 7 — Data Cleaning", "evidence": "cleaning_summary + *_clean DataFrames with standardized statuses, dates, and numeric fields.", "completion_status": "Complete"},
    {"prd_requirement": "FR-05 Analyze order delays", "notebook_section": "Section 8 — Order Delay Detection", "evidence": "delivery_delay_days, delay_bucket, delay_summary, city_delay_summary.", "completion_status": "Complete"},
    {"prd_requirement": "FR-06 Analyze refunds", "notebook_section": "Section 9 — Refund Delay and Refund-Priority Analysis", "evidence": "refunds_analysis with pending/failed/SLA breach/mismatch/priority flags, refund_summary.", "completion_status": "Complete"},
    {"prd_requirement": "FR-07 Link complaints", "notebook_section": "Section 10 — Customer Complaint Linking", "evidence": "complaints_by_order aggregation with sentiment and repeated-complaint flags.", "completion_status": "Complete"},
    {"prd_requirement": "FR-08 Map support tickets", "notebook_section": "Section 11 — Support Ticket Mapping", "evidence": "tickets_order aggregation with escalation and SLA breach detection.", "completion_status": "Complete"},
    {"prd_requirement": "FR-09 Detect duplicate/repeat issues", "notebook_section": "Section 12 — Duplicate and Repeat Issue Detection", "evidence": "duplicate_summary covering duplicate IDs, duplicate-looking orders, repeated complaints/tickets.", "completion_status": "Complete"},
    {"prd_requirement": "FR-10 Build final feature table", "notebook_section": "Section 13 — Final Feature Table / Master Merge", "evidence": "final_feature_table built by aggregating before merging; merge_validation shows no row multiplication.", "completion_status": "Complete"},
    {"prd_requirement": "FR-11 Classify priority", "notebook_section": "Section 14 — Priority Classification", "evidence": "classify_issue_priority() produces issue_score/issue_priority/priority_reasons; priority_summary.", "completion_status": "Complete"},
    {"prd_requirement": "FR-12 Generate recommended action", "notebook_section": "Section 15 — Recommended Action Generator", "evidence": "generate_recommended_action() produces specific, signal-driven recommended_action values.", "completion_status": "Complete"},
    {"prd_requirement": "FR-13 Generate AI-ready prompts", "notebook_section": "Section 16 — AI-Ready Prompt Generator", "evidence": "generate_ai_prompt() builds fact-only ai_prompt_preview text; no external API call anywhere.", "completion_status": "Complete"},
    {"prd_requirement": "FR-14 Build Colab GUI", "notebook_section": "Section 17 — Colab GUI/Search/Filter Experience", "evidence": "ipywidgets priority/city/search filters with fallback static preview if widgets are unavailable.", "completion_status": "Complete"},
    {"prd_requirement": "FR-15 Export final report", "notebook_section": "Section 18 — Final Report Export and Preview", "evidence": "export_final_report() writes final_issue_resolution_report.csv; schema/shape/preview shown.", "completion_status": "Complete"},
    {"prd_requirement": "FR-16 Run validation checks", "notebook_section": "Section 19 — Final Validation Checks", "evidence": "final_validation_df with 12 checks and a final success/failure message.", "completion_status": "Complete"},
    {"prd_requirement": "BR-01 Delivery delay from actual vs promised date", "notebook_section": "Section 8", "evidence": "calculate_delay_days() uses actual_delivery_date and promised_delivery_date.", "completion_status": "Complete"},
    {"prd_requirement": "BR-02 Pending/failed refund on delayed order = refund-priority", "notebook_section": "Section 9", "evidence": "refund_priority_candidate flag combines refund status with delay_bucket.", "completion_status": "Complete"},
    {"prd_requirement": "BR-03 Refund SLA breach > 7 days", "notebook_section": "Section 9", "evidence": "refund_sla_breach uses REFUND_SLA_DAYS=7 against refund_age_days / refund_completion_delay_days.", "completion_status": "Complete"},
    {"prd_requirement": "BR-04 Complaint sentiment raises customer impact", "notebook_section": "Sections 10 & 14", "evidence": "angry/negative complaint counts feed directly into issue_score.", "completion_status": "Complete"},
    {"prd_requirement": "BR-05 Escalated/long-resolution tickets = operational risk", "notebook_section": "Sections 11 & 14", "evidence": "ticket_escalated and ticket_sla_breach (24h threshold) feed into issue_score.", "completion_status": "Complete"},
    {"prd_requirement": "BR-06 Repeated complaints flagged", "notebook_section": "Sections 10, 12 & 14", "evidence": "repeated_complaint_flag and repeated_complaint_customers; contributes to issue_score.", "completion_status": "Complete"},
    {"prd_requirement": "BR-07 Invalid data flagged, not deleted", "notebook_section": "Section 6", "evidence": "validation_summary flags issues; no rows are dropped anywhere in the notebook.", "completion_status": "Complete"},
    {"prd_requirement": "BR-08 Priority combines multiple signals", "notebook_section": "Section 14", "evidence": "classify_issue_priority() combines delay, refund, complaint, ticket, and segment signals, not delay alone.", "completion_status": "Complete"},
])
display(prd_completion_mapping)


,prd_requirement,notebook_section,evidence,completion_status
0,FR-01 Load dataset ZIP in Colab,Dataset Upload Instructions (pre-Section 1),ZIP extraction cell creates DATA_DIR and lists...,Complete
1,FR-02 Load all data files,Section 5 — Multi-File DataLoader,FlipkartIssueDataLoader.load_all() loads CSV/E...,Complete
2,FR-03 Validate data integrity,Section 6 — Data Validation,"validation_summary reports required columns, d...",Complete
3,FR-04 Clean and standardize data,Section 7 — Data Cleaning,cleaning_summary + *_clean DataFrames with sta...,Complete
4,FR-05 Analyze order delays,Section 8 — Order Delay Detection,"delivery_delay_days, delay_bucket, delay_summa...",Complete
5,FR-06 Analyze refunds,Section 9 — Refund Delay and Refund-Priority A...,refunds_analysis with pending/failed/SLA breac...,Complete
6,FR-07 Link complaints,Section 10 — Customer Complaint Linking,complaints_by_order aggregation with sentiment...,Complete
7,FR-08 Map support tickets,Section 11 — Support Ticket Mapping,tickets_order aggregation with escalation and ...,Complete
8,FR-09 Detect duplicate/repeat issues,Section 12 — Duplicate and Repeat Issue Detection,"duplicate_summary covering duplicate IDs, dupl...",Complete
9,FR-10 Build final feature table,Section 13 — Final Feature Table / Master Merge,final_feature_table built by aggregating befor...,Complete


## Assumption and Limitation Log

Document assumptions, edge cases, and implementation limitations.

In [24]:
# DELIVERABLE — Assumption and Limitation Log

assumption_limitation_log = pd.DataFrame([
    {"type": "Assumption", "description": f"Reference 'current date' for pending refund age and pending-delivery delay is {REFERENCE_DATE.date()} (one day after the latest date seen across all datasets).", "impact": "Pending-case age/delay figures shift if the true business snapshot date differs.", "mitigation": "REFERENCE_DATE is a single named constant in Section 1 that can be changed in one place."},
    {"type": "Assumption", "description": "Delay buckets: On Time <=0 days, Minor 1-2 days, Moderate 3-5 days, Severe >5 days; missing/invalid promised date -> 'Delivery Date Missing or Invalid'.", "impact": "Bucket boundaries are a business judgment call, not stated numerically in the PRD.", "mitigation": "Thresholds are defined in classify_delay_bucket() and documented in Section 8's markdown/comments."},
    {"type": "Assumption", "description": "Ticket SLA threshold is 24 hours, per the inherited notebook hint; escalated tickets with missing resolution_time_hours are treated as an SLA breach.", "impact": "Different SLA policies would change ticket_sla_breach_count and downstream priority scores.", "mitigation": "TICKET_SLA_HOURS is a named constant in Section 1."},
    {"type": "Assumption", "description": "Refund SLA breach threshold is 7 days for both pending-refund age and initiated-to-completed delay, per BR-03.", "impact": "A different SLA policy would change refund_sla_breach flags.", "mitigation": "REFUND_SLA_DAYS is a named constant in Section 1."},
    {"type": "Assumption", "description": "issue_score point values (e.g. Severe Delay=30, Failed Refund=20, Escalated Ticket=20) are an implementation decision, not specified numerically in the PRD.", "impact": "Absolute score values and Critical/High/Medium/Low cut points (70/40/15) are illustrative, not calibrated against real operations data.", "mitigation": "All point values and thresholds are visible and commented in classify_issue_priority() (Section 14) for easy tuning."},
    {"type": "Limitation", "description": "Invalid or orphan records (e.g. refunds pointing to a missing order_id) are flagged in validation but are still included wherever they can be safely processed; they are never silently dropped, per BR-07.", "impact": "Some aggregate counts include records tied to data-quality issues that a real production system might exclude or route for manual review.", "mitigation": "validation_summary and duplicate_summary make every such record explicitly visible and countable."},
    {"type": "Limitation", "description": "Duplicate-looking orders (same customer/seller/date/amount) are flagged, not deduplicated, since it isn't certain they are true duplicates rather than legitimate repeat purchases.", "impact": "final_feature_table may contain more than one row for what could be the same real-world order.", "mitigation": "duplicate_summary (Section 12) reports the count so an operations reviewer can decide manually."},
    {"type": "Limitation", "description": "All data is synthetic and generated for this educational exercise only; it does not represent real Flipkart customers, sellers, or orders.", "impact": "None for grading purposes; this notebook must never be presented as analyzing real company data.", "mitigation": "Stated explicitly here and in the notebook's opening markdown cells."},
    {"type": "Limitation", "description": "The Colab GUI (Section 17) uses ipywidgets, which renders interactively only inside a live Jupyter/Colab kernel; a static HTML export or PDF will only show the last rendered filter state.", "impact": "Reviewers opening a static export may not be able to interact with the filters.", "mitigation": "A non-widget fallback preview (High-priority cases) is printed automatically if ipywidgets fails to load."},
])
display(assumption_limitation_log)


,type,description,impact,mitigation
0,Assumption,Reference 'current date' for pending refund ag...,Pending-case age/delay figures shift if the tr...,REFERENCE_DATE is a single named constant in S...
1,Assumption,"Delay buckets: On Time <=0 days, Minor 1-2 day...",Bucket boundaries are a business judgment call...,Thresholds are defined in classify_delay_bucke...
2,Assumption,"Ticket SLA threshold is 24 hours, per the inhe...",Different SLA policies would change ticket_sla...,TICKET_SLA_HOURS is a named constant in Sectio...
3,Assumption,Refund SLA breach threshold is 7 days for both...,A different SLA policy would change refund_sla...,REFUND_SLA_DAYS is a named constant in Section 1.
4,Assumption,issue_score point values (e.g. Severe Delay=30...,Absolute score values and Critical/High/Medium...,All point values and thresholds are visible an...
5,Limitation,Invalid or orphan records (e.g. refunds pointi...,Some aggregate counts include records tied to ...,validation_summary and duplicate_summary make ...
6,Limitation,Duplicate-looking orders (same customer/seller...,final_feature_table may contain more than one ...,duplicate_summary (Section 12) reports the cou...
7,Limitation,All data is synthetic and generated for this e...,None for grading purposes; this notebook must ...,Stated explicitly here and in the notebook's o...
8,Limitation,"The Colab GUI (Section 17) uses ipywidgets, wh...",Reviewers opening a static export may not be a...,A non-widget fallback preview (High-priority c...


## Final Report Preview Evidence

Show report file name, shape, schema, and high-priority preview.

In [25]:
# DELIVERABLE — Final Report Preview Evidence

final_report_preview_evidence = {
    "report_filename": REPORT_PATH,
    "report_shape": final_report.shape,
    "report_schema": list(final_report.columns),
    "priority_summary": priority_summary.set_index("issue_priority")["order_count"].to_dict(),
}

print("Report filename:", final_report_preview_evidence["report_filename"])
print("Report shape:", final_report_preview_evidence["report_shape"])
print("Report schema:", final_report_preview_evidence["report_schema"])
print("Priority summary:", final_report_preview_evidence["priority_summary"])
print()
print("Critical / High priority case preview:")
display(final_report[final_report["issue_priority"].isin(["Critical", "High"])]
        .sort_values("issue_score", ascending=False).head(10))


Report filename: final_issue_resolution_report.csv
Report shape: (5000, 18)
Report schema: ['order_id', 'customer_id', 'seller_id', 'delivery_city', 'order_amount', 'order_status_clean', 'delivery_delay_days', 'delay_bucket', 'refund_count', 'pending_refund_count', 'failed_refund_count', 'complaint_count', 'ticket_count', 'issue_score', 'issue_priority', 'recommended_action', 'ai_prompt_preview', 'report_generated_at']
Priority summary: {'Critical': 149, 'High': 903, 'Medium': 2199, 'Low': 1749}

Critical / High priority case preview:


,order_id,customer_id,seller_id,delivery_city,order_amount,order_status_clean,delivery_delay_days,delay_bucket,refund_count,pending_refund_count,failed_refund_count,complaint_count,ticket_count,issue_score,issue_priority,recommended_action,ai_prompt_preview,report_generated_at
3695,ORD0003696,CUST000395,SELL0307,Chennai,59530.80,Processing,13.0,Severe Delay,3,1,1,0,1,125,Critical,Refund reconciliation: investigate failed refu...,"Order ORD0003696 (customer CUST000395, seller ...",2026-06-30
2686,ORD0002687,CUST000018,SELL0553,Kochi,23047.07,Processing,52.0,Severe Delay,2,1,0,2,1,120,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0002687 (customer CUST000018, seller ...",2026-06-30
749,ORD0000750,CUST000431,SELL0567,Mumbai,40173.12,Delayed,9.0,Severe Delay,2,1,1,2,1,115,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0000750 (customer CUST000431, seller ...",2026-06-30
421,ORD0000422,CUST000756,SELL0957,Mumbai,46937.94,Processing,16.0,Severe Delay,2,1,0,0,1,110,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0000422 (customer CUST000756, seller ...",2026-06-30
1843,ORD0001844,CUST000082,SELL0330,Hyderabad,57748.72,Processing,45.0,Severe Delay,2,1,1,1,0,100,Critical,Refund reconciliation: investigate failed refu...,"Order ORD0001844 (customer CUST000082, seller ...",2026-06-30
2360,ORD0002361,CUST000786,SELL0882,Lucknow,64394.30,Processing,24.0,Severe Delay,1,1,0,0,2,100,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0002361 (customer CUST000786, seller ...",2026-06-30
690,ORD0000691,CUST000486,SELL0940,Chennai,63283.36,Processing,14.0,Severe Delay,1,1,0,0,1,100,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0000691 (customer CUST000486, seller ...",2026-06-30
1096,ORD0001097,CUST000173,SELL0770,Ahmedabad,50010.47,Returned,9.0,Severe Delay,1,0,1,0,1,100,Critical,Refund operations review: failed refund needs ...,"Order ORD0001097 (customer CUST000173, seller ...",2026-06-30
1376,ORD0001377,CUST001143,SELL0473,Coimbatore,63556.61,Delayed,9.0,Severe Delay,1,1,0,0,1,100,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0001377 (customer CUST001143, seller ...",2026-06-30
1076,ORD0001077,CUST000048,SELL0879,Delhi,39059.43,Cancelled,22.0,Severe Delay,1,1,0,0,1,95,Critical,Refund operations follow-up: pending refund ha...,"Order ORD0001077 (customer CUST000048, seller ...",2026-06-30


## Final Product Walkthrough

Explain the completed internal tool as if handing it back to Product and Engineering.

In [26]:
# DELIVERABLE — Final Product Walkthrough

final_product_walkthrough = [
    "1. Data loading: FlipkartIssueDataLoader (Section 5) reads orders/customers/sellers as CSV, refunds as Excel, "
    "complaints as JSON, and status notes as TXT, with exception handling so one bad file doesn't crash the run.",
    "2. Quality detection: Section 6 checks required columns, duplicate IDs, orphan order/customer/seller/refund/"
    "complaint/ticket links, invalid dates, and refund amount anomalies — every issue is counted and reported, never silently dropped.",
    "3. Cleaning: Section 7 standardizes text casing/whitespace and safely parses dates and numeric fields into "
    "*_clean columns while keeping the original raw values for auditability.",
    "4. Delay analysis: Section 8 compares actual vs promised delivery dates (and estimates current delay for still-"
    "pending orders) and buckets every order into On Time / Minor / Moderate / Severe / Missing-Invalid.",
    "5. Refund, complaint, and ticket signals: Sections 9-11 aggregate each operational file to one row per order "
    "BEFORE merging, detecting pending/failed refunds, SLA breaches, amount mismatches, angry/negative complaints, "
    "repeated complaints, escalated tickets, and ticket SLA breaches.",
    "6. Master table: Section 13 merges customer, seller, and the three aggregated signal tables onto the cleaned "
    "orders table with left joins, and merge_validation proves the row count never multiplies.",
    "7. Priority and actions: Section 14 scores every order from all of the above signals into Critical/High/Medium/"
    "Low with visible reasons; Section 15 turns those same signals into a specific recommended action instead of a generic message.",
    "8. Search, export, and validation: Section 17 gives operations users a priority/city/search filter GUI; Section "
    "18 exports the full report as final_issue_resolution_report.csv; Section 19 runs 12 checks and prints a clear pass/fail result.",
]
for line in final_product_walkthrough:
    print(line)


1. Data loading: FlipkartIssueDataLoader (Section 5) reads orders/customers/sellers as CSV, refunds as Excel, complaints as JSON, and status notes as TXT, with exception handling so one bad file doesn't crash the run.
2. Quality detection: Section 6 checks required columns, duplicate IDs, orphan order/customer/seller/refund/complaint/ticket links, invalid dates, and refund amount anomalies — every issue is counted and reported, never silently dropped.
3. Cleaning: Section 7 standardizes text casing/whitespace and safely parses dates and numeric fields into *_clean columns while keeping the original raw values for auditability.
4. Delay analysis: Section 8 compares actual vs promised delivery dates (and estimates current delay for still-pending orders) and buckets every order into On Time / Minor / Moderate / Severe / Missing-Invalid.
5. Refund, complaint, and ticket signals: Sections 9-11 aggregate each operational file to one row per order BEFORE merging, detecting pending/failed refu

## Final Self-Check

Use this as your final readiness check before submission.

In [27]:
# DELIVERABLE — Final Self-Check

final_self_check = pd.DataFrame([
    {"check_item": "Dataset ZIP uploaded and extracted", "status": "PASS" if Path(DATA_DIR).exists() else "FAIL", "notes": f"DATA_DIR={DATA_DIR}"},
    {"check_item": "All 7 raw files loaded", "status": "PASS" if (dataset_load_summary["status"] == "Loaded").all() else "FAIL", "notes": "See dataset_load_summary in Section 5."},
    {"check_item": "Validation checks executed and visible", "status": "PASS" if len(validation_summary) > 0 else "FAIL", "notes": f"{len(validation_summary)} checks in validation_summary."},
    {"check_item": "Cleaning applied without deleting records", "status": "PASS" if len(orders_clean) == len(all_data['orders']) else "FAIL", "notes": f"orders_clean rows={len(orders_clean)} vs raw orders={len(all_data['orders'])}."},
    {"check_item": "Delay detection uses actual_delivery_date", "status": "PASS" if "delivery_delay_days" in orders_clean.columns else "FAIL", "notes": "See Section 8."},
    {"check_item": "Refund priority logic implemented", "status": "PASS" if "refund_priority_candidate" in refunds_analysis.columns else "FAIL", "notes": "See Section 9."},
    {"check_item": "Complaints aggregated before merge", "status": "PASS" if "complaint_count" in final_feature_table.columns else "FAIL", "notes": "See Sections 10 & 13."},
    {"check_item": "Tickets aggregated before merge", "status": "PASS" if "ticket_count" in final_feature_table.columns else "FAIL", "notes": "See Sections 11 & 13."},
    {"check_item": "Final feature table has no row explosion", "status": "PASS" if merge_validation["row_count"].nunique() == 1 else "FAIL", "notes": f"merge_validation row counts: {merge_validation['row_count'].tolist()}."},
    {"check_item": "Priority combines multiple signals (BR-08)", "status": "PASS" if final_feature_table["priority_reasons"].str.contains(";").any() else "WARNING", "notes": "priority_reasons shows multiple contributing signals for many orders."},
    {"check_item": "Recommended actions are specific, not generic", "status": "PASS" if final_feature_table["recommended_action"].nunique() > 5 else "WARNING", "notes": f"{final_feature_table['recommended_action'].nunique()} distinct action combinations."},
    {"check_item": "AI prompts generated without any API call", "status": "PASS", "notes": "generate_ai_prompt() only formats text; no network/API calls exist anywhere in the notebook."},
    {"check_item": "Colab GUI present with fallback", "status": "PASS", "notes": "Section 17 uses ipywidgets with a static fallback preview."},
    {"check_item": "Final report exported with full schema", "status": "PASS" if set(REPORT_SCHEMA).issubset(final_report.columns) else "FAIL", "notes": f"Report path: {REPORT_PATH}."},
    {"check_item": "Final validation checks all passed", "status": "PASS" if final_validation_df["passed"].all() else "FAIL", "notes": f"{int(final_validation_df['passed'].sum())}/{len(final_validation_df)} checks passed."},
])
display(final_self_check)


,check_item,status,notes
0,Dataset ZIP uploaded and extracted,PASS,DATA_DIR=flipkart_order_issue_resolution_dataset
1,All 7 raw files loaded,PASS,See dataset_load_summary in Section 5.
2,Validation checks executed and visible,PASS,26 checks in validation_summary.
3,Cleaning applied without deleting records,PASS,orders_clean rows=5000 vs raw orders=5000.
4,Delay detection uses actual_delivery_date,PASS,See Section 8.
5,Refund priority logic implemented,PASS,See Section 9.
6,Complaints aggregated before merge,PASS,See Sections 10 & 13.
7,Tickets aggregated before merge,PASS,See Sections 11 & 13.
8,Final feature table has no row explosion,PASS,"merge_validation row counts: [5000, 5000, 5000..."
9,Priority combines multiple signals (BR-08),PASS,priority_reasons shows multiple contributing s...


## Presentation and Explanation Notes

Prepare a short explanation for reviewers/stakeholders.

In [28]:
# DELIVERABLE — Presentation and Explanation Notes

presentation_explanation_notes = """
Business problem: Flipkart support/refund/seller/city operations teams need one place that connects
order delays, refund status, customer complaints, and support tickets, instead of checking four
separate files, so that the most urgent cases surface quickly.

Users: Support executives and team leads (find/triage urgent cases), refund operations (prioritize
follow-up), seller operations (spot seller-level friction), and city operations (spot city-level
friction) — see PRD section 4.

Data sources: orders.csv, customers.csv, sellers.csv (CSV), refunds.xlsx (Excel), customer_complaints.json
(JSON), support_tickets.csv (CSV), status_notes.txt (TXT) — all synthetic, loaded by
FlipkartIssueDataLoader with per-file exception handling.

Processing flow: load -> validate (flag, never silently delete) -> clean/standardize -> detect delivery
delays -> analyze refund risk -> aggregate complaints -> aggregate tickets -> aggregate BEFORE merging into
one order-level final_feature_table -> score priority -> generate a recommended action -> generate an
AI-ready prompt (text only, no API call) -> filter/search via GUI -> export final CSV -> validate output.

Business rules applied: BR-01 delay from actual vs promised date; BR-02 pending/failed refund on a
delayed order is refund-priority; BR-03 refund SLA breach at 7 days; BR-04 angry/negative sentiment
raises customer impact; BR-05 escalated/long-resolution tickets are operational risk (24h SLA);
BR-06 repeated complaints are flagged; BR-07 invalid data is flagged, never deleted; BR-08 priority
combines multiple signals, never delay alone.

Priority scoring: classify_issue_priority() adds documented points for delay severity, refund
signals, complaint signals, ticket signals, and customer segment, then maps the total score to
Critical / High / Medium / Low with a human-readable priority_reasons explanation per order.

GUI: Section 17 offers priority, city, and order/customer-ID search filters using ipywidgets, with a
static fallback preview if widgets can't render, so the notebook still works as a lightweight
internal tool even outside a live Colab session.

Final report: export_final_report() writes final_issue_resolution_report.csv containing the full
required schema (order/customer/seller context, delay info, refund/complaint/ticket counts, priority,
recommended action, and AI prompt preview) plus a generated timestamp.

Validation and limitations: Section 19 runs 12 automated checks (schema, nulls, row counts, read-back)
before declaring success. Key assumptions — the reference "current date", delay-bucket thresholds, and
SLA thresholds — are documented explicitly in the Assumption and Limitation Log rather than hidden in
code, since they are implementation decisions rather than facts stated numerically in the PRD.
"""
print(presentation_explanation_notes)



Business problem: Flipkart support/refund/seller/city operations teams need one place that connects
order delays, refund status, customer complaints, and support tickets, instead of checking four
separate files, so that the most urgent cases surface quickly.

Users: Support executives and team leads (find/triage urgent cases), refund operations (prioritize
follow-up), seller operations (spot seller-level friction), and city operations (spot city-level
friction) — see PRD section 4.

Data sources: orders.csv, customers.csv, sellers.csv (CSV), refunds.xlsx (Excel), customer_complaints.json
(JSON), support_tickets.csv (CSV), status_notes.txt (TXT) — all synthetic, loaded by
FlipkartIssueDataLoader with per-file exception handling.

Processing flow: load -> validate (flag, never silently delete) -> clean/standardize -> detect delivery
delays -> analyze refund risk -> aggregate complaints -> aggregate tickets -> aggregate BEFORE merging into
one order-level final_feature_table -> score pri

## 100-Mark Evaluation Rubric

| Criteria | Marks |
|---|---:|
| Data loading and file handling for CSV, Excel, JSON, TXT, and ZIP extraction | 8 |
| Data validation and relationship quality checks | 10 |
| Data cleaning and standardization of statuses, dates, text, and amounts | 10 |
| Correct use of Python functions and modular code | 10 |
| OOP implementation and code organization | 8 |
| Core business logic for delays, refunds, complaints, tickets, and priority scoring | 12 |
| Pandas analysis and aggregations for operations insights | 10 |
| Colab GUI/search/filter experience | 8 |
| Report generation and notebook-visible outputs | 7 |
| Debug log, AI usage log, assumptions, and PRD mapping | 7 |
| Presentation & Explanation | 10 |
| **Total** | **100** |

## Final Submission Checklist

Before submitting the completed notebook, confirm:

- [ ] Dataset ZIP was uploaded and extracted successfully.
- [ ] All code cells run from top to bottom in Google Colab.
- [ ] All major section outputs are visible.
- [ ] Debug Fix Log is filled inside the notebook.
- [ ] AI Prompt Usage Log is filled inside the notebook.
- [ ] PRD Completion Mapping is filled inside the notebook.
- [ ] Assumption and Limitation Log is filled inside the notebook.
- [ ] Final Report Preview Evidence is visible.
- [ ] Final Product Walkthrough is complete.
- [ ] Final Self-Check is complete.
- [ ] Presentation and Explanation Notes are complete.
- [ ] The only final deliverable is the completed Colab notebook with all outputs visible.